# 🎵 Riffusion Kaggle Song Studio

## Notebook-as-a-Service: Remote-Controlled Long-Form AI Music Generation

This notebook sets up a FastAPI server that exposes the Riffusion song generation pipeline,
accessible via a Cloudflare tunnel from your desktop/mobile app.

### Features:
- **5-10 Minute Songs**: Parses lyrics into structured sections (Intro, Verse, Chorus, Bridge, Outro)
- **Seamless Stitching**: Crossfades and normalizes audio for professional results
- **Checkpointing**: Resume generation if notebook disconnects
- **REST API**: Full remote control via HTTP endpoints
- **No Cost**: Uses free Kaggle T4 GPU resources

### Quick Start:
1. Run all cells in order
2. Copy the public tunnel URL
3. Use the client SDK or REST API to generate songs

## Step 1: Install Dependencies

In [ ]:
# Install system dependencies
!apt-get update -qq
!apt-get install -y -qq ffmpeg

# Install Python packages
!pip install -q \
    torch torchaudio \
    transformers diffusers accelerate \
    pydub librosa soundfile \
    fastapi uvicorn python-multipart \
    pyyaml requests pillow numpy scipy

print("✅ Dependencies installed")

## Step 2: Install Cloudflared for Tunnel

In [ ]:
# Download and install cloudflared
!curl -L --output /tmp/cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i /tmp/cloudflared.deb

print("✅ Cloudflared installed")

## Step 3: Setup Source Package

In [ ]:
# ── generated by scripts/build-riffusion-notebook.py ──
# Regenerate with: python3 scripts/build-riffusion-notebook.py
#
# The source package is embedded because a Kaggle push carries exactly one file. Writing it
# out here is what makes `import src.api_server` below work at all.
import base64, os, pathlib, sys

NOTEBOOK_DIR = '/kaggle/working/kaggle_riffusion/notebook'

_BUNDLE = {
    'src/__init__.py': (
        'IiIiClJpZmZ1c2lvbiBLYWdnbGUgU29uZyBTdHVkaW8gLSBOb3RlYm9vayBTb3VyY2UgUGFja2FnZQoiIiIKCl9fdmVyc2lv'
        'bl9fID0gIjEuMC4wIgpfX2F1dGhvcl9fID0gIkxpZ2h0a2lkIEFJIFN0dWRpbyIK'
    ),
    'src/api_server.py': (
        '...'
        'YWluX18iOgogICAgc3RhcnRfc2VydmVyKCkK'
    ),
    'src/audio_stitcher.py': (
        'IiIiCkF1ZGlvIFN0aXRjaGluZyBNb2R1bGUgZm9yIFJpZmZ1c2lvbiBTb25nIFN0dWRpbwoKSGFuZGxlcyBzZWFtbGVzcyBj'
        'b25jYXRlbmF0aW9uIG9mIGF1ZGlvIGNsaXBzIHdpdGggY3Jvc3NmYWRlcywKYmVhdC1tYXRjaGluZyBhc3Npc3RhbmNlLCBh'
        'bmQgbG91ZG5lc3Mgbm9ybWFsaXphdGlvbi4KIiIiCgppbXBvcnQgb3MKaW1wb3J0IHRlbXBmaWxlCmZyb20gcGF0aGxpYiBp'
        'bXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgTGlzdCwgT3B0aW9uYWwsIFR1cGxlCmZyb20gZGF0YWNsYXNzZXMgaW1w'
        'b3J0IGRhdGFjbGFzcwoKdHJ5OgogICAgZnJvbSBweWR1YiBpbXBvcnQgQXVkaW9TZWdtZW50CiAgICBmcm9tIHB5ZHViLmVm'
        'ZmVjdHMgaW1wb3J0IG5vcm1hbGl6ZQpleGNlcHQgSW1wb3J0RXJyb3I6CiAgICBBdWRpb1NlZ21lbnQgPSBOb25lICAjIHR5'
        'cGU6IGlnbm9yZQogICAgbm9ybWFsaXplID0gTm9uZSAgIyB0eXBlOiBpZ25vcmUKCmltcG9ydCBsb2dnaW5nCgpsb2dnZXIg'
        'PSBsb2dnaW5nLmdldExvZ2dlcihfX25hbWVfXykKCgpAZGF0YWNsYXNzCmNsYXNzIFN0aXRjaENvbmZpZzoKICAgICIiIkNv'
        'bmZpZ3VyYXRpb24gZm9yIGF1ZGlvIHN0aXRjaGluZyBvcGVyYXRpb25zLiIiIgogICAgY3Jvc3NmYWRlX2R1cmF0aW9uOiBm'
        'bG9hdCA9IDEuMCAgIyBzZWNvbmRzCiAgICBub3JtYWxpemVfdGFyZ2V0X2RiZnM6IGZsb2F0ID0gLTE2LjAgICMgTFVGUyB0'
        'YXJnZXQgZm9yIHN0cmVhbWluZwogICAgZmFkZV9vdXRfZHVyYXRpb246IGZsb2F0ID0gMi4wICAjIHNlY29uZHMgZm9yIGZp'
        'bmFsIGZhZGUgb3V0CiAgICBzYW1wbGVfcmF0ZTogaW50ID0gNDQxMDAKICAgIGNoYW5uZWxzOiBpbnQgPSAyCiAgICBmb3Jt'
        'YXQ6IHN0ciA9ICJ3YXYiCgoKZGVmIGxvYWRfYXVkaW9fY2xpcChmaWxlX3BhdGg6IHN0cikgLT4gIkF1ZGlvU2VnbWVudCI6'
        'CiAgICAiIiJMb2FkIGFuIGF1ZGlvIGNsaXAgZnJvbSBmaWxlLiIiIgogICAgaWYgQXVkaW9TZWdtZW50IGlzIE5vbmU6CiAg'
        'ICAgICAgcmFpc2UgSW1wb3J0RXJyb3IoInB5ZHViIG5vdCBpbnN0YWxsZWQuIEluc3RhbGwgd2l0aDogcGlwIGluc3RhbGwg'
        'cHlkdWIiKQogICAgCiAgICBwYXRoID0gUGF0aChmaWxlX3BhdGgpCiAgICBpZiBub3QgcGF0aC5leGlzdHMoKToKICAgICAg'
        'ICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmIkF1ZGlvIGZpbGUgbm90IGZvdW5kOiB7ZmlsZV9wYXRofSIpCiAgICAKICAg'
        'ICMgRGV0ZXJtaW5lIGZvcm1hdCBmcm9tIGV4dGVuc2lvbgogICAgZXh0ID0gcGF0aC5zdWZmaXgubG93ZXIoKS5sc3RyaXAo'
        'Jy4nKQogICAgcmV0dXJuIEF1ZGlvU2VnbWVudC5mcm9tX2ZpbGUoc3RyKHBhdGgpLCBmb3JtYXQ9ZXh0IG9yICJ3YXYiKQoK'
        'CmRlZiBhcHBseV9jcm9zc2ZhZGUoCiAgICBjbGlwMTogIkF1ZGlvU2VnbWVudCIsCiAgICBjbGlwMjogIkF1ZGlvU2VnbWVu'
        'dCIsCiAgICBjcm9zc2ZhZGVfZHVyYXRpb246IGZsb2F0ID0gMS4wCikgLT4gIkF1ZGlvU2VnbWVudCI6CiAgICAiIiIKICAg'
        'IEFwcGx5IGNyb3NzZmFkZSBiZXR3ZWVuIHR3byBhdWRpbyBjbGlwcy4KICAgIAogICAgQXJnczoKICAgICAgICBjbGlwMTog'
        'Rmlyc3QgYXVkaW8gc2VnbWVudAogICAgICAgIGNsaXAyOiBTZWNvbmQgYXVkaW8gc2VnbWVudCAgCiAgICAgICAgY3Jvc3Nm'
        'YWRlX2R1cmF0aW9uOiBEdXJhdGlvbiBvZiBjcm9zc2ZhZGUgaW4gc2Vjb25kcwogICAgCiAgICBSZXR1cm5zOgogICAgICAg'
        'IENvbWJpbmVkIGF1ZGlvIHNlZ21lbnQgd2l0aCBjcm9zc2ZhZGUKICAgICIiIgogICAgaWYgQXVkaW9TZWdtZW50IGlzIE5v'
        'bmU6CiAgICAgICAgcmFpc2UgSW1wb3J0RXJyb3IoInB5ZHViIG5vdCBpbnN0YWxsZWQiKQogICAgCiAgICAjIENvbnZlcnQg'
        'Y3Jvc3NmYWRlIGR1cmF0aW9uIHRvIG1pbGxpc2Vjb25kcwogICAgY3Jvc3NmYWRlX21zID0gaW50KGNyb3NzZmFkZV9kdXJh'
        'dGlvbiAqIDEwMDApCiAgICAKICAgICMgRW5zdXJlIGNyb3NzZmFkZSBkb2Vzbid0IGV4Y2VlZCBjbGlwIGxlbmd0aHMKICAg'
        'IGNyb3NzZmFkZV9tcyA9IG1pbihjcm9zc2ZhZGVfbXMsIGxlbihjbGlwMSkgLSAxMDAsIGxlbihjbGlwMikgLSAxMDApCiAg'
        'ICBjcm9zc2ZhZGVfbXMgPSBtYXgoY3Jvc3NmYWRlX21zLCAwKQogICAgCiAgICAjIE92ZXJsYXkgY2xpcDIgb250byBjbGlw'
        'MSB3aXRoIGZhZGUgaW4vb3V0CiAgICBjbGlwMl9mYWRlZCA9IGNsaXAyLmZhZGVfaW4oY3Jvc3NmYWRlX21zKS5mYWRlX291'
        'dChjcm9zc2ZhZGVfbXMpCiAgICAKICAgICMgQ2FsY3VsYXRlIHBvc2l0aW9uIHRvIHN0YXJ0IG92ZXJsYXkKICAgIG92ZXJs'
        'YXlfcG9zaXRpb24gPSBsZW4oY2xpcDEpIC0gY3Jvc3NmYWRlX21zCiAgICAKICAgIGNvbWJpbmVkID0gY2xpcDEub3Zlcmxh'
        'eShjbGlwMl9mYWRlZCwgcG9zaXRpb249aW50KG92ZXJsYXlfcG9zaXRpb24pKQogICAgCiAgICBsb2dnZXIuZGVidWcoZiJD'
        'cm9zc2ZhZGVkIGNsaXBzOiB7bGVuKGNsaXAxKX1tcyArIHtsZW4oY2xpcDIpfW1zIHdpdGgge2Nyb3NzZmFkZV9tc31tcyBv'
        'dmVybGFwIikKICAgIAogICAgcmV0dXJuIGNvbWJpbmVkCgoKZGVmIGNvbmNhdGVuYXRlX2NsaXBzKAogICAgY2xpcHM6IExp'
        'c3RbIkF1ZGlvU2VnbWVudCJdLAogICAgY3Jvc3NmYWRlX2R1cmF0aW9uOiBmbG9hdCA9IDEuMAopIC0+ICJBdWRpb1NlZ21l'
        'bnQiOgogICAgIiIiCiAgICBDb25jYXRlbmF0ZSBtdWx0aXBsZSBhdWRpbyBjbGlwcyB3aXRoIGNyb3NzZmFkZXMuCiAgICAK'
        'ICAgIEFyZ3M6CiAgICAgICAgY2xpcHM6IExpc3Qgb2YgYXVkaW8gc2VnbWVudHMgdG8gY29uY2F0ZW5hdGUKICAgICAgICBj'
        'cm9zc2ZhZGVfZHVyYXRpb246IER1cmF0aW9uIG9mIGNyb3NzZmFkZSBiZXR3ZWVuIGNsaXBzCiAgICAKICAgIFJldHVybnM6'
        'CiAgICAgICAgU2luZ2xlIGNvbmNhdGVuYXRlZCBhdWRpbyBzZWdtZW50CiAgICAiIiIKICAgIGlmIG5vdCBjbGlwczoKICAg'
        'ICAgICByYWlzZSBWYWx1ZUVycm9yKCJObyBjbGlwcyBwcm92aWRlZCBmb3IgY29uY2F0ZW5hdGlvbiIpCiAgICAKICAgIGlm'
        'IGxlbihjbGlwcykgPT0gMToKICAgICAgICByZXR1cm4gY2xpcHNbMF0KICAgIAogICAgcmVzdWx0ID0gY2xpcHNbMF0KICAg'
        'IGZvciBpLCBjbGlwIGluIGVudW1lcmF0ZShjbGlwc1sxOl0sIHN0YXJ0PTEpOgogICAgICAgIHJlc3VsdCA9IGFwcGx5X2Ny'
        'b3NzZmFkZShyZXN1bHQsIGNsaXAsIGNyb3NzZmFkZV9kdXJhdGlvbikKICAgICAgICBsb2dnZXIuZGVidWcoZiJDb25jYXRl'
        'bmF0ZWQgY2xpcCB7aX0ve2xlbihjbGlwcykgLSAxfSIpCiAgICAKICAgIHJldHVybiByZXN1bHQKCgpkZWYgbm9ybWFsaXpl'
        'X2xvdWRuZXNzKAogICAgYXVkaW86ICJBdWRpb1NlZ21lbnQiLAogICAgdGFyZ2V0X2RiZnM6IGZsb2F0ID0gLTE2LjAKKSAt'
        'PiAiQXVkaW9TZWdtZW50IjoKICAgICIiIgogICAgTm9ybWFsaXplIGF1ZGlvIGxvdWRuZXNzIHRvIHRhcmdldCBkQkZTLgog'
        'ICAgCiAgICBBcmdzOgogICAgICAgIGF1ZGlvOiBJbnB1dCBhdWRpbyBzZWdtZW50CiAgICAgICAgdGFyZ2V0X2RiZnM6IFRh'
        'cmdldCBsb3VkbmVzcyBpbiBkQkZTIChkZWZhdWx0IC0xNiBmb3Igc3RyZWFtaW5nKQogICAgCiAgICBSZXR1cm5zOgogICAg'
        'ICAgIE5vcm1hbGl6ZWQgYXVkaW8gc2VnbWVudAogICAgIiIiCiAgICBpZiBub3JtYWxpemUgaXMgTm9uZToKICAgICAgICBy'
        'YWlzZSBJbXBvcnRFcnJvcigicHlkdWIuZWZmZWN0cy5ub3JtYWxpemUgbm90IGF2YWlsYWJsZSIpCiAgICAKICAgICMgU2lt'
        'cGxlIHBlYWsgbm9ybWFsaXphdGlvbiB1c2luZyBweWR1YgogICAgY2hhbmdlX2luX2RiZnMgPSB0YXJnZXRfZGJmcyAtIGF1'
        'ZGlvLmRCRlMKICAgIG5vcm1hbGl6ZWQgPSBhdWRpby5hcHBseV9nYWluKGNoYW5nZV9pbl9kYmZzKQogICAgCiAgICAjIENs'
        'aXAgdG8gcHJldmVudCBkaWdpdGFsIGNsaXBwaW5nCiAgICBub3JtYWxpemVkID0gbm9ybWFsaXplZC5saW1pdCgwLjApCiAg'
        'ICAKICAgIGxvZ2dlci5pbmZvKGYiTm9ybWFsaXplZCBhdWRpbyBmcm9tIHthdWRpby5kQkZTOi4xZn0gZEJGUyB0byB7dGFy'
        'Z2V0X2RiZnM6LjFmfSBkQkZTIikKICAgIAogICAgcmV0dXJuIG5vcm1hbGl6ZWQKCgpkZWYgYXBwbHlfZmluYWxfZmFkZV9v'
        'dXQoCiAgICBhdWRpbzogIkF1ZGlvU2VnbWVudCIsCiAgICBmYWRlX2R1cmF0aW9uOiBmbG9hdCA9IDIuMAopIC0+ICJBdWRp'
        'b1NlZ21lbnQiOgogICAgIiIiCiAgICBBcHBseSBhIGZhZGUtb3V0IGVmZmVjdCB0byB0aGUgZW5kIG9mIGFuIGF1ZGlvIHRy'
        'YWNrLgogICAgCiAgICBBcmdzOgogICAgICAgIGF1ZGlvOiBJbnB1dCBhdWRpbyBzZWdtZW50CiAgICAgICAgZmFkZV9kdXJh'
        'dGlvbjogRHVyYXRpb24gb2YgZmFkZS1vdXQgaW4gc2Vjb25kcwogICAgCiAgICBSZXR1cm5zOgogICAgICAgIEF1ZGlvIHNl'
        'Z21lbnQgd2l0aCBmYWRlLW91dCBhcHBsaWVkCiAgICAiIiIKICAgIGZhZGVfbXMgPSBpbnQoZmFkZV9kdXJhdGlvbiAqIDEw'
        'MDApCiAgICBmYWRlX21zID0gbWluKGZhZGVfbXMsIGxlbihhdWRpbykgLSAxMDApICAjIERvbid0IGZhZGUgZW50aXJlIHRy'
        'YWNrCiAgICAKICAgIHJldHVybiBhdWRpby5mYWRlX291dChmYWRlX21zKQoKCmRlZiBzdGl0Y2hfc29uZ19zZWN0aW9ucygK'
        'ICAgIHNlY3Rpb25fZmlsZXM6IExpc3Rbc3RyXSwKICAgIGNvbmZpZzogT3B0aW9uYWxbU3RpdGNoQ29uZmlnXSA9IE5vbmUs'
        'CiAgICBvdXRwdXRfcGF0aDogT3B0aW9uYWxbc3RyXSA9IE5vbmUKKSAtPiBzdHI6CiAgICAiIiIKICAgIFN0aXRjaCB0b2dl'
        'dGhlciBzb25nIHNlY3Rpb25zIGludG8gYSBjb21wbGV0ZSBzb25nLgogICAgCiAgICBBcmdzOgogICAgICAgIHNlY3Rpb25f'
        'ZmlsZXM6IExpc3Qgb2YgcGF0aHMgdG8gc2VjdGlvbiBhdWRpbyBmaWxlcyAoaW4gb3JkZXIpCiAgICAgICAgY29uZmlnOiBT'
        'dGl0Y2hpbmcgY29uZmlndXJhdGlvbgogICAgICAgIG91dHB1dF9wYXRoOiBPcHRpb25hbCBvdXRwdXQgZmlsZSBwYXRoLiBJ'
        'ZiBOb25lLCB1c2VzIHRlbXAgZmlsZS4KICAgIAogICAgUmV0dXJuczoKICAgICAgICBQYXRoIHRvIHRoZSBzdGl0Y2hlZCBh'
        'dWRpbyBmaWxlCiAgICAiIiIKICAgIGlmIEF1ZGlvU2VnbWVudCBpcyBOb25lOgogICAgICAgIHJhaXNlIEltcG9ydEVycm9y'
        'KCJweWR1YiBub3QgaW5zdGFsbGVkLiBJbnN0YWxsIHdpdGg6IHBpcCBpbnN0YWxsIHB5ZHViIikKICAgIAogICAgY29uZmln'
        'ID0gY29uZmlnIG9yIFN0aXRjaENvbmZpZygpCiAgICAKICAgIGlmIG5vdCBzZWN0aW9uX2ZpbGVzOgogICAgICAgIHJhaXNl'
        'IFZhbHVlRXJyb3IoIk5vIHNlY3Rpb24gZmlsZXMgcHJvdmlkZWQiKQogICAgCiAgICBsb2dnZXIuaW5mbyhmIlN0aXRjaGlu'
        'ZyB7bGVuKHNlY3Rpb25fZmlsZXMpfSBzZWN0aW9ucy4uLiIpCiAgICAKICAgICMgTG9hZCBhbGwgY2xpcHMKICAgIGNsaXBz'
        'ID0gW10KICAgIGZvciBpLCBmaWxlX3BhdGggaW4gZW51bWVyYXRlKHNlY3Rpb25fZmlsZXMpOgogICAgICAgIHRyeToKICAg'
        'ICAgICAgICAgY2xpcCA9IGxvYWRfYXVkaW9fY2xpcChmaWxlX3BhdGgpCiAgICAgICAgICAgIAogICAgICAgICAgICAjIEVu'
        'c3VyZSBjb25zaXN0ZW50IHNhbXBsZSByYXRlIGFuZCBjaGFubmVscwogICAgICAgICAgICBjbGlwID0gY2xpcC5zZXRfZnJh'
        'bWVfcmF0ZShjb25maWcuc2FtcGxlX3JhdGUpCiAgICAgICAgICAgIGNsaXAgPSBjbGlwLnNldF9jaGFubmVscyhjb25maWcu'
        'Y2hhbm5lbHMpCiAgICAgICAgICAgIAogICAgICAgICAgICBjbGlwcy5hcHBlbmQoY2xpcCkKICAgICAgICAgICAgbG9nZ2Vy'
        'LmRlYnVnKGYiTG9hZGVkIHNlY3Rpb24ge2kgKyAxfToge2xlbihjbGlwKSAvIDEwMDA6LjFmfXMiKQogICAgICAgIGV4Y2Vw'
        'dCBFeGNlcHRpb24gYXMgZToKICAgICAgICAgICAgbG9nZ2VyLmVycm9yKGYiRmFpbGVkIHRvIGxvYWQgc2VjdGlvbiB7aSAr'
        'IDF9ICh7ZmlsZV9wYXRofSk6IHtlfSIpCiAgICAgICAgICAgIHJhaXNlCiAgICAKICAgICMgQ29uY2F0ZW5hdGUgd2l0aCBj'
        'cm9zc2ZhZGVzCiAgICBjb21iaW5lZCA9IGNvbmNhdGVuYXRlX2NsaXBzKGNsaXBzLCBjb25maWcuY3Jvc3NmYWRlX2R1cmF0'
        'aW9uKQogICAgCiAgICAjIE5vcm1hbGl6ZSBsb3VkbmVzcwogICAgbm9ybWFsaXplZCA9IG5vcm1hbGl6ZV9sb3VkbmVzcyhj'
        'b21iaW5lZCwgY29uZmlnLm5vcm1hbGl6ZV90YXJnZXRfZGJmcykKICAgIAogICAgIyBBcHBseSBmaW5hbCBmYWRlIG91dAog'
        'ICAgZmluYWwgPSBhcHBseV9maW5hbF9mYWRlX291dChub3JtYWxpemVkLCBjb25maWcuZmFkZV9vdXRfZHVyYXRpb24pCiAg'
        'ICAKICAgICMgRGV0ZXJtaW5lIG91dHB1dCBwYXRoCiAgICBpZiBvdXRwdXRfcGF0aCBpcyBOb25lOgogICAgICAgIGZkLCBv'
        'dXRwdXRfcGF0aCA9IHRlbXBmaWxlLm1rc3RlbXAoc3VmZml4PWYiLntjb25maWcuZm9ybWF0fSIpCiAgICAgICAgb3MuY2xv'
        'c2UoZmQpCiAgICAKICAgICMgRXhwb3J0IGZpbmFsIGF1ZGlvCiAgICBmaW5hbC5leHBvcnQoCiAgICAgICAgb3V0cHV0X3Bh'
        'dGgsCiAgICAgICAgZm9ybWF0PWNvbmZpZy5mb3JtYXQsCiAgICAgICAgcGFyYW1ldGVycz1bIi1xOmEiLCAiMCJdIGlmIGNv'
        'bmZpZy5mb3JtYXQgPT0gIm1wMyIgZWxzZSBbXQogICAgKQogICAgCiAgICB0b3RhbF9kdXJhdGlvbiA9IGxlbihmaW5hbCkg'
        'LyAxMDAwCiAgICBsb2dnZXIuaW5mbyhmIlN0aXRjaGVkIHNvbmcgY29tcGxldGU6IHt0b3RhbF9kdXJhdGlvbjouMWZ9cyBz'
        'YXZlZCB0byB7b3V0cHV0X3BhdGh9IikKICAgIAogICAgcmV0dXJuIG91dHB1dF9wYXRoCgoKZGVmIGVzdGltYXRlX3RvdGFs'
        'X2R1cmF0aW9uKAogICAgc2VjdGlvbl9kdXJhdGlvbnM6IExpc3RbZmxvYXRdLAogICAgY3Jvc3NmYWRlX2R1cmF0aW9uOiBm'
        'bG9hdCA9IDEuMAopIC0+IGZsb2F0OgogICAgIiIiCiAgICBFc3RpbWF0ZSB0b3RhbCBkdXJhdGlvbiBhZnRlciBzdGl0Y2hp'
        'bmcgd2l0aCBjcm9zc2ZhZGVzLgogICAgCiAgICBBcmdzOgogICAgICAgIHNlY3Rpb25fZHVyYXRpb25zOiBMaXN0IG9mIHNl'
        'Y3Rpb24gZHVyYXRpb25zIGluIHNlY29uZHMKICAgICAgICBjcm9zc2ZhZGVfZHVyYXRpb246IENyb3NzZmFkZSBkdXJhdGlv'
        'biBiZXR3ZWVuIHNlY3Rpb25zCiAgICAKICAgIFJldHVybnM6CiAgICAgICAgRXN0aW1hdGVkIHRvdGFsIGR1cmF0aW9uIGlu'
        'IHNlY29uZHMKICAgICIiIgogICAgaWYgbm90IHNlY3Rpb25fZHVyYXRpb25zOgogICAgICAgIHJldHVybiAwLjAKICAgIAog'
        'ICAgdG90YWwgPSBzdW0oc2VjdGlvbl9kdXJhdGlvbnMpCiAgICAjIFN1YnRyYWN0IG92ZXJsYXAgdGltZSAob25lIGxlc3Mg'
        'Y3Jvc3NmYWRlIHRoYW4gc2VjdGlvbnMpCiAgICBvdmVybGFwID0gY3Jvc3NmYWRlX2R1cmF0aW9uICogKGxlbihzZWN0aW9u'
        'X2R1cmF0aW9ucykgLSAxKQogICAgCiAgICByZXR1cm4gdG90YWwgLSBvdmVybGFwCgoKZGVmIGdldF9hdWRpb19pbmZvKGZp'
        'bGVfcGF0aDogc3RyKSAtPiBkaWN0OgogICAgIiIiCiAgICBHZXQgaW5mb3JtYXRpb24gYWJvdXQgYW4gYXVkaW8gZmlsZS4K'
        'ICAgIAogICAgQXJnczoKICAgICAgICBmaWxlX3BhdGg6IFBhdGggdG8gYXVkaW8gZmlsZQogICAgCiAgICBSZXR1cm5zOgog'
        'ICAgICAgIERpY3Rpb25hcnkgd2l0aCBhdWRpbyBtZXRhZGF0YQogICAgIiIiCiAgICBpZiBBdWRpb1NlZ21lbnQgaXMgTm9u'
        'ZToKICAgICAgICByYWlzZSBJbXBvcnRFcnJvcigicHlkdWIgbm90IGluc3RhbGxlZCIpCiAgICAKICAgIGF1ZGlvID0gbG9h'
        'ZF9hdWRpb19jbGlwKGZpbGVfcGF0aCkKICAgIAogICAgcmV0dXJuIHsKICAgICAgICAiZHVyYXRpb25fc2Vjb25kcyI6IGxl'
        'bihhdWRpbykgLyAxMDAwLAogICAgICAgICJzYW1wbGVfcmF0ZSI6IGF1ZGlvLmZyYW1lX3JhdGUsCiAgICAgICAgImNoYW5u'
        'ZWxzIjogYXVkaW8uY2hhbm5lbHMsCiAgICAgICAgImJpdF9kZXB0aCI6IGF1ZGlvLnNhbXBsZV93aWR0aCAqIDgsCiAgICAg'
        'ICAgImRCRlMiOiBhdWRpby5kQkZTLAogICAgICAgICJmaWxlX3NpemVfYnl0ZXMiOiBvcy5wYXRoLmdldHNpemUoZmlsZV9w'
        'YXRoKQogICAgfQo='
    ),
    'src/job_queue.py': (
        'IiIiCkpvYiBRdWV1ZSBNb2R1bGUgZm9yIFJpZmZ1c2lvbiBLYWdnbGUgU29uZyBTdHVkaW8KCk1hbmFnZXMgYXN5bmMgam9i'
        'IHF1ZXVlIHdpdGggc3RhdHVzIHRyYWNraW5nIGFuZCBwZXJzaXN0ZW5jZS4KIiIiCgppbXBvcnQgdGhyZWFkaW5nCmZyb20g'
        'ZGF0ZXRpbWUgaW1wb3J0IGRhdGV0aW1lCmZyb20gdHlwaW5nIGltcG9ydCBEaWN0LCBMaXN0LCBPcHRpb25hbCwgQW55CmZy'
        'b20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcywgZmllbGQKZnJvbSBlbnVtIGltcG9ydCBFbnVtCmltcG9ydCBsb2dn'
        'aW5nCgpsb2dnZXIgPSBsb2dnaW5nLmdldExvZ2dlcihfX25hbWVfXykKCgpjbGFzcyBKb2JTdGF0dXMoc3RyLCBFbnVtKToK'
        'ICAgICIiIlN0YXR1cyB2YWx1ZXMgZm9yIGdlbmVyYXRpb24gam9icy4iIiIKICAgIFFVRVVFRCA9ICJxdWV1ZWQiCiAgICBQ'
        'Uk9DRVNTSU5HID0gInByb2Nlc3NpbmciCiAgICBDT01QTEVURUQgPSAiY29tcGxldGVkIgogICAgRkFJTEVEID0gImZhaWxl'
        'ZCIKICAgIENBTkNFTExFRCA9ICJjYW5jZWxsZWQiCgoKQGRhdGFjbGFzcwpjbGFzcyBKb2I6CiAgICAiIiJSZXByZXNlbnRz'
        'IGEgc29uZyBnZW5lcmF0aW9uIGpvYi4iIiIKICAgIGpvYl9pZDogc3RyCiAgICB0aXRsZTogc3RyCiAgICBzdGF0dXM6IEpv'
        'YlN0YXR1cyA9IEpvYlN0YXR1cy5RVUVVRUQKICAgIGNyZWF0ZWRfYXQ6IGZsb2F0ID0gZmllbGQoZGVmYXVsdF9mYWN0b3J5'
        'PWxhbWJkYTogZGF0ZXRpbWUubm93KCkudGltZXN0YW1wKCkpCiAgICBzdGFydGVkX2F0OiBPcHRpb25hbFtmbG9hdF0gPSBO'
        'b25lCiAgICBjb21wbGV0ZWRfYXQ6IE9wdGlvbmFsW2Zsb2F0XSA9IE5vbmUKICAgIHBhcmFtczogRGljdFtzdHIsIEFueV0g'
        'PSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9ZGljdCkKICAgIG91dHB1dF9maWxlOiBPcHRpb25hbFtzdHJdID0gTm9uZQogICAg'
        'ZXJyb3JfbWVzc2FnZTogT3B0aW9uYWxbc3RyXSA9IE5vbmUKICAgIHBhcmVudF9qb2JfaWQ6IE9wdGlvbmFsW3N0cl0gPSBO'
        'b25lICAjIEZvciBhbHRlcm5hdGUgdmVyc2lvbnMKICAgIAogICAgZGVmIHRvX2RpY3Qoc2VsZikgLT4gZGljdDoKICAgICAg'
        'ICAiIiJDb252ZXJ0IHRvIGRpY3Rpb25hcnkgZm9yIHNlcmlhbGl6YXRpb24uIiIiCiAgICAgICAgcmV0dXJuIHsKICAgICAg'
        'ICAgICAgImpvYl9pZCI6IHNlbGYuam9iX2lkLAogICAgICAgICAgICAidGl0bGUiOiBzZWxmLnRpdGxlLAogICAgICAgICAg'
        'ICAic3RhdHVzIjogc2VsZi5zdGF0dXMudmFsdWUsCiAgICAgICAgICAgICJjcmVhdGVkX2F0Ijogc2VsZi5jcmVhdGVkX2F0'
        'LAogICAgICAgICAgICAic3RhcnRlZF9hdCI6IHNlbGYuc3RhcnRlZF9hdCwKICAgICAgICAgICAgImNvbXBsZXRlZF9hdCI6'
        'IHNlbGYuY29tcGxldGVkX2F0LAogICAgICAgICAgICAicGFyYW1zIjogc2VsZi5wYXJhbXMsCiAgICAgICAgICAgICJvdXRw'
        'dXRfZmlsZSI6IHNlbGYub3V0cHV0X2ZpbGUsCiAgICAgICAgICAgICJlcnJvcl9tZXNzYWdlIjogc2VsZi5lcnJvcl9tZXNz'
        'YWdlLAogICAgICAgICAgICAicGFyZW50X2pvYl9pZCI6IHNlbGYucGFyZW50X2pvYl9pZAogICAgICAgIH0KCgpjbGFzcyBK'
        'b2JRdWV1ZToKICAgICIiIgogICAgVGhyZWFkLXNhZmUgam9iIHF1ZXVlIGZvciBtYW5hZ2luZyBzb25nIGdlbmVyYXRpb24g'
        'dGFza3MuCiAgICAKICAgIFByb3ZpZGVzIGJhc2ljIHF1ZXVlIG9wZXJhdGlvbnMgYW5kIHN0YXR1cyB0cmFja2luZy4KICAg'
        'IEluIHByb2R1Y3Rpb24sIGNvdWxkIGJlIHJlcGxhY2VkIHdpdGggUmVkaXMvQ2VsZXJ5LgogICAgIiIiCiAgICAKICAgIGRl'
        'ZiBfX2luaXRfXyhzZWxmLCBtYXhfY29uY3VycmVudDogaW50ID0gMSk6CiAgICAgICAgIiIiCiAgICAgICAgSW5pdGlhbGl6'
        'ZSB0aGUgam9iIHF1ZXVlLgogICAgICAgIAogICAgICAgIEFyZ3M6CiAgICAgICAgICAgIG1heF9jb25jdXJyZW50OiBNYXhp'
        'bXVtIGNvbmN1cnJlbnQgam9icyAoS2FnZ2xlIGxpbWl0ZWQgdG8gMSkKICAgICAgICAiIiIKICAgICAgICBzZWxmLmpvYnM6'
        'IERpY3Rbc3RyLCBKb2JdID0ge30KICAgICAgICBzZWxmLnF1ZXVlOiBMaXN0W3N0cl0gPSBbXSAgIyBKb2IgSURzIGluIHF1'
        'ZXVlIG9yZGVyCiAgICAgICAgc2VsZi5tYXhfY29uY3VycmVudCA9IG1heF9jb25jdXJyZW50CiAgICAgICAgc2VsZi5hY3Rp'
        'dmVfam9iczogc2V0ID0gc2V0KCkKICAgICAgICBzZWxmLl9sb2NrID0gdGhyZWFkaW5nLkxvY2soKQogICAgICAgIAogICAg'
        'ICAgIGxvZ2dlci5pbmZvKGYiSm9iUXVldWUgaW5pdGlhbGl6ZWQgd2l0aCBtYXhfY29uY3VycmVudD17bWF4X2NvbmN1cnJl'
        'bnR9IikKICAgIAogICAgZGVmIGFkZF9qb2Ioc2VsZiwgam9iOiBKb2IsIHBhcmFtczogRGljdFtzdHIsIEFueV0pIC0+IHN0'
        'cjoKICAgICAgICAiIiIKICAgICAgICBBZGQgYSBqb2IgdG8gdGhlIHF1ZXVlLgogICAgICAgIAogICAgICAgIEFyZ3M6CiAg'
        'ICAgICAgICAgIGpvYjogSm9iIG9iamVjdAogICAgICAgICAgICBwYXJhbXM6IEdlbmVyYXRpb24gcGFyYW1ldGVycwogICAg'
        'ICAgIAogICAgICAgIFJldHVybnM6CiAgICAgICAgICAgIEpvYiBJRAogICAgICAgICIiIgogICAgICAgIHdpdGggc2VsZi5f'
        'bG9jazoKICAgICAgICAgICAgam9iLnBhcmFtcyA9IHBhcmFtcwogICAgICAgICAgICBzZWxmLmpvYnNbam9iLmpvYl9pZF0g'
        'PSBqb2IKICAgICAgICAgICAgc2VsZi5xdWV1ZS5hcHBlbmQoam9iLmpvYl9pZCkKICAgICAgICAgICAgCiAgICAgICAgICAg'
        'IGxvZ2dlci5kZWJ1ZyhmIkFkZGVkIGpvYiB7am9iLmpvYl9pZH0gdG8gcXVldWUiKQogICAgICAgIAogICAgICAgIHJldHVy'
        'biBqb2Iuam9iX2lkCiAgICAKICAgIGRlZiBnZXRfam9iKHNlbGYsIGpvYl9pZDogc3RyKSAtPiBPcHRpb25hbFtKb2JdOgog'
        'ICAgICAgICIiIkdldCBhIGpvYiBieSBJRC4iIiIKICAgICAgICByZXR1cm4gc2VsZi5qb2JzLmdldChqb2JfaWQpCiAgICAK'
        'ICAgIGRlZiBnZXRfbmV4dF9qb2Ioc2VsZikgLT4gT3B0aW9uYWxbSm9iXToKICAgICAgICAiIiIKICAgICAgICBHZXQgdGhl'
        'IG5leHQgcXVldWVkIGpvYiBpZiBjYXBhY2l0eSBhdmFpbGFibGUuCiAgICAgICAgCiAgICAgICAgUmV0dXJuczoKICAgICAg'
        'ICAgICAgTmV4dCBqb2Igb3IgTm9uZSBpZiBxdWV1ZSBlbXB0eSBvciBhdCBjYXBhY2l0eQogICAgICAgICIiIgogICAgICAg'
        'IHdpdGggc2VsZi5fbG9jazoKICAgICAgICAgICAgaWYgbGVuKHNlbGYuYWN0aXZlX2pvYnMpID49IHNlbGYubWF4X2NvbmN1'
        'cnJlbnQ6CiAgICAgICAgICAgICAgICByZXR1cm4gTm9uZQogICAgICAgICAgICAKICAgICAgICAgICAgd2hpbGUgc2VsZi5x'
        'dWV1ZToKICAgICAgICAgICAgICAgIGpvYl9pZCA9IHNlbGYucXVldWUucG9wKDApCiAgICAgICAgICAgICAgICBqb2IgPSBz'
        'ZWxmLmpvYnMuZ2V0KGpvYl9pZCkKICAgICAgICAgICAgICAgIAogICAgICAgICAgICAgICAgaWYgam9iIGFuZCBqb2Iuc3Rh'
        'dHVzID09IEpvYlN0YXR1cy5RVUVVRUQ6CiAgICAgICAgICAgICAgICAgICAgc2VsZi5hY3RpdmVfam9icy5hZGQoam9iX2lk'
        'KQogICAgICAgICAgICAgICAgICAgIHJldHVybiBqb2IKICAgICAgICAgICAgCiAgICAgICAgICAgIHJldHVybiBOb25lCiAg'
        'ICAKICAgIGRlZiBtYXJrX3N0YXJ0ZWQoc2VsZiwgam9iX2lkOiBzdHIpIC0+IE5vbmU6CiAgICAgICAgIiIiTWFyayBhIGpv'
        'YiBhcyBzdGFydGVkLiIiIgogICAgICAgIHdpdGggc2VsZi5fbG9jazoKICAgICAgICAgICAgam9iID0gc2VsZi5qb2JzLmdl'
        'dChqb2JfaWQpCiAgICAgICAgICAgIGlmIGpvYjoKICAgICAgICAgICAgICAgIGpvYi5zdGF0dXMgPSBKb2JTdGF0dXMuUFJP'
        'Q0VTU0lORwogICAgICAgICAgICAgICAgam9iLnN0YXJ0ZWRfYXQgPSBkYXRldGltZS5ub3coKS50aW1lc3RhbXAoKQogICAg'
        'CiAgICBkZWYgbWFya19jb21wbGV0ZWQoc2VsZiwgam9iX2lkOiBzdHIsIG91dHB1dF9maWxlOiBPcHRpb25hbFtzdHJdID0g'
        'Tm9uZSkgLT4gTm9uZToKICAgICAgICAiIiJNYXJrIGEgam9iIGFzIGNvbXBsZXRlZC4iIiIKICAgICAgICB3aXRoIHNlbGYu'
        'X2xvY2s6CiAgICAgICAgICAgIGpvYiA9IHNlbGYuam9icy5nZXQoam9iX2lkKQogICAgICAgICAgICBpZiBqb2I6CiAgICAg'
        'ICAgICAgICAgICBqb2Iuc3RhdHVzID0gSm9iU3RhdHVzLkNPTVBMRVRFRAogICAgICAgICAgICAgICAgam9iLmNvbXBsZXRl'
        'ZF9hdCA9IGRhdGV0aW1lLm5vdygpLnRpbWVzdGFtcCgpCiAgICAgICAgICAgICAgICBqb2Iub3V0cHV0X2ZpbGUgPSBvdXRw'
        'dXRfZmlsZQogICAgICAgICAgICAgICAgCiAgICAgICAgICAgICAgICBpZiBqb2JfaWQgaW4gc2VsZi5hY3RpdmVfam9iczoK'
        'ICAgICAgICAgICAgICAgICAgICBzZWxmLmFjdGl2ZV9qb2JzLnJlbW92ZShqb2JfaWQpCiAgICAKICAgIGRlZiBtYXJrX2Zh'
        'aWxlZChzZWxmLCBqb2JfaWQ6IHN0ciwgZXJyb3I6IHN0cikgLT4gTm9uZToKICAgICAgICAiIiJNYXJrIGEgam9iIGFzIGZh'
        'aWxlZC4iIiIKICAgICAgICB3aXRoIHNlbGYuX2xvY2s6CiAgICAgICAgICAgIGpvYiA9IHNlbGYuam9icy5nZXQoam9iX2lk'
        'KQogICAgICAgICAgICBpZiBqb2I6CiAgICAgICAgICAgICAgICBqb2Iuc3RhdHVzID0gSm9iU3RhdHVzLkZBSUxFRAogICAg'
        'ICAgICAgICAgICAgam9iLmNvbXBsZXRlZF9hdCA9IGRhdGV0aW1lLm5vdygpLnRpbWVzdGFtcCgpCiAgICAgICAgICAgICAg'
        'ICBqb2IuZXJyb3JfbWVzc2FnZSA9IGVycm9yCiAgICAgICAgICAgICAgICAKICAgICAgICAgICAgICAgIGlmIGpvYl9pZCBp'
        'biBzZWxmLmFjdGl2ZV9qb2JzOgogICAgICAgICAgICAgICAgICAgIHNlbGYuYWN0aXZlX2pvYnMucmVtb3ZlKGpvYl9pZCkK'
        'ICAgIAogICAgZGVmIGNhbmNlbF9qb2Ioc2VsZiwgam9iX2lkOiBzdHIpIC0+IGJvb2w6CiAgICAgICAgIiIiCiAgICAgICAg'
        'Q2FuY2VsIGEgcXVldWVkIG9yIHByb2Nlc3Npbmcgam9iLgogICAgICAgIAogICAgICAgIFJldHVybnM6CiAgICAgICAgICAg'
        'IFRydWUgaWYgY2FuY2VsbGVkLCBGYWxzZSBpZiBub3QgY2FuY2VsbGFibGUKICAgICAgICAiIiIKICAgICAgICB3aXRoIHNl'
        'bGYuX2xvY2s6CiAgICAgICAgICAgIGpvYiA9IHNlbGYuam9icy5nZXQoam9iX2lkKQogICAgICAgICAgICBpZiBub3Qgam9i'
        'OgogICAgICAgICAgICAgICAgcmV0dXJuIEZhbHNlCiAgICAgICAgICAgIAogICAgICAgICAgICBpZiBqb2Iuc3RhdHVzIGlu'
        'IFtKb2JTdGF0dXMuQ09NUExFVEVELCBKb2JTdGF0dXMuRkFJTEVEXToKICAgICAgICAgICAgICAgIHJldHVybiBGYWxzZQog'
        'ICAgICAgICAgICAKICAgICAgICAgICAgam9iLnN0YXR1cyA9IEpvYlN0YXR1cy5DQU5DRUxMRUQKICAgICAgICAgICAgam9i'
        'LmNvbXBsZXRlZF9hdCA9IGRhdGV0aW1lLm5vdygpLnRpbWVzdGFtcCgpCiAgICAgICAgICAgIAogICAgICAgICAgICBpZiBq'
        'b2JfaWQgaW4gc2VsZi5hY3RpdmVfam9iczoKICAgICAgICAgICAgICAgIHNlbGYuYWN0aXZlX2pvYnMucmVtb3ZlKGpvYl9p'
        'ZCkKICAgICAgICAgICAgCiAgICAgICAgICAgICMgUmVtb3ZlIGZyb20gcXVldWUgaWYgc3RpbGwgcXVldWVkCiAgICAgICAg'
        'ICAgIGlmIGpvYl9pZCBpbiBzZWxmLnF1ZXVlOgogICAgICAgICAgICAgICAgc2VsZi5xdWV1ZS5yZW1vdmUoam9iX2lkKQog'
        'ICAgICAgICAgICAKICAgICAgICAgICAgcmV0dXJuIFRydWUKICAgIAogICAgZGVmIGdldF9xdWV1ZV9zdGF0dXMoc2VsZikg'
        'LT4gZGljdDoKICAgICAgICAiIiJHZXQgY3VycmVudCBxdWV1ZSBzdGF0dXMuIiIiCiAgICAgICAgd2l0aCBzZWxmLl9sb2Nr'
        'OgogICAgICAgICAgICBzdGF0dXNfY291bnRzID0ge30KICAgICAgICAgICAgZm9yIGpvYiBpbiBzZWxmLmpvYnMudmFsdWVz'
        'KCk6CiAgICAgICAgICAgICAgICBzdGF0dXMgPSBqb2Iuc3RhdHVzLnZhbHVlCiAgICAgICAgICAgICAgICBzdGF0dXNfY291'
        'bnRzW3N0YXR1c10gPSBzdGF0dXNfY291bnRzLmdldChzdGF0dXMsIDApICsgMQogICAgICAgICAgICAKICAgICAgICAgICAg'
        'cmV0dXJuIHsKICAgICAgICAgICAgICAgICJ0b3RhbF9qb2JzIjogbGVuKHNlbGYuam9icyksCiAgICAgICAgICAgICAgICAi'
        'cXVldWVkIjogbGVuKHNlbGYucXVldWUpLAogICAgICAgICAgICAgICAgImFjdGl2ZSI6IGxlbihzZWxmLmFjdGl2ZV9qb2Jz'
        'KSwKICAgICAgICAgICAgICAgICJieV9zdGF0dXMiOiBzdGF0dXNfY291bnRzCiAgICAgICAgICAgIH0KICAgIAogICAgZGVm'
        'IGxpc3Rfam9icyhzZWxmLCBzdGF0dXNfZmlsdGVyOiBPcHRpb25hbFtKb2JTdGF0dXNdID0gTm9uZSkgLT4gTGlzdFtKb2Jd'
        'OgogICAgICAgICIiIkxpc3Qgam9icywgb3B0aW9uYWxseSBmaWx0ZXJlZCBieSBzdGF0dXMuIiIiCiAgICAgICAgd2l0aCBz'
        'ZWxmLl9sb2NrOgogICAgICAgICAgICBqb2JzID0gbGlzdChzZWxmLmpvYnMudmFsdWVzKCkpCiAgICAgICAgICAgIAogICAg'
        'ICAgICAgICBpZiBzdGF0dXNfZmlsdGVyOgogICAgICAgICAgICAgICAgam9icyA9IFtqIGZvciBqIGluIGpvYnMgaWYgai5z'
        'dGF0dXMgPT0gc3RhdHVzX2ZpbHRlcl0KICAgICAgICAgICAgCiAgICAgICAgICAgICMgU29ydCBieSBjcmVhdGVkX2F0IGRl'
        'c2NlbmRpbmcKICAgICAgICAgICAgam9icy5zb3J0KGtleT1sYW1iZGEgajogai5jcmVhdGVkX2F0LCByZXZlcnNlPVRydWUp'
        'CiAgICAgICAgICAgIAogICAgICAgICAgICByZXR1cm4gam9icwogICAgCiAgICBkZWYgY2xlYW51cF9vbGRfam9icyhzZWxm'
        'LCBtYXhfYWdlX2hvdXJzOiBpbnQgPSAyNCkgLT4gaW50OgogICAgICAgICIiIgogICAgICAgIFJlbW92ZSBvbGQgY29tcGxl'
        'dGVkL2ZhaWxlZC9jYW5jZWxsZWQgam9icy4KICAgICAgICAKICAgICAgICBBcmdzOgogICAgICAgICAgICBtYXhfYWdlX2hv'
        'dXJzOiBNYXhpbXVtIGFnZSBpbiBob3VycwogICAgICAgIAogICAgICAgIFJldHVybnM6CiAgICAgICAgICAgIE51bWJlciBv'
        'ZiBqb2JzIHJlbW92ZWQKICAgICAgICAiIiIKICAgICAgICBjdXRvZmYgPSBkYXRldGltZS5ub3coKS50aW1lc3RhbXAoKSAt'
        'IChtYXhfYWdlX2hvdXJzICogMzYwMCkKICAgICAgICByZW1vdmVkID0gMAogICAgICAgIAogICAgICAgIHdpdGggc2VsZi5f'
        'bG9jazoKICAgICAgICAgICAgdG9fcmVtb3ZlID0gW10KICAgICAgICAgICAgCiAgICAgICAgICAgIGZvciBqb2JfaWQsIGpv'
        'YiBpbiBzZWxmLmpvYnMuaXRlbXMoKToKICAgICAgICAgICAgICAgIGlmIGpvYi5zdGF0dXMgaW4gW0pvYlN0YXR1cy5DT01Q'
        'TEVURUQsIEpvYlN0YXR1cy5GQUlMRUQsIEpvYlN0YXR1cy5DQU5DRUxMRURdOgogICAgICAgICAgICAgICAgICAgIGlmIGpv'
        'Yi5jb21wbGV0ZWRfYXQgYW5kIGpvYi5jb21wbGV0ZWRfYXQgPCBjdXRvZmY6CiAgICAgICAgICAgICAgICAgICAgICAgIHRv'
        'X3JlbW92ZS5hcHBlbmQoam9iX2lkKQogICAgICAgICAgICAKICAgICAgICAgICAgZm9yIGpvYl9pZCBpbiB0b19yZW1vdmU6'
        'CiAgICAgICAgICAgICAgICBkZWwgc2VsZi5qb2JzW2pvYl9pZF0KICAgICAgICAgICAgICAgIGlmIGpvYl9pZCBpbiBzZWxm'
        'LnF1ZXVlOgogICAgICAgICAgICAgICAgICAgIHNlbGYucXVldWUucmVtb3ZlKGpvYl9pZCkKICAgICAgICAgICAgICAgIHJl'
        'bW92ZWQgKz0gMQogICAgICAgIAogICAgICAgIGxvZ2dlci5pbmZvKGYiQ2xlYW5lZCB1cCB7cmVtb3ZlZH0gb2xkIGpvYnMi'
        'KQogICAgICAgIHJldHVybiByZW1vdmVkCiAgICAKICAgIGRlZiBfX2xlbl9fKHNlbGYpIC0+IGludDoKICAgICAgICAiIiJS'
        'ZXR1cm4gdG90YWwgbnVtYmVyIG9mIGpvYnMuIiIiCiAgICAgICAgcmV0dXJuIGxlbihzZWxmLmpvYnMpCg=='
    ),
    'src/long_form_generator.py': (
        '...'
    ),
    'src/preset_engine.py': (
        'IiIiClByZXNldCBFbmdpbmUgZm9yIFJpZmZ1c2lvbiBTb25nIFN0dWRpbwoKTWFuYWdlcyBtdXNpY2FsIHN0eWxlIHByZXNl'
        'dHMgYW5kIHNlY3Rpb24tc3BlY2lmaWMgcHJvbXB0IG1vZGlmaWNhdGlvbnMuCkxvYWRzIGNvbmZpZ3VyYXRpb24gZnJvbSBZ'
        'QU1MIGZpbGVzIGFuZCBwcm92aWRlcyBwcm9tcHQgc3ludGhlc2lzIHV0aWxpdGllcy4KIiIiCgppbXBvcnQgb3MKZnJvbSBw'
        'YXRobGliIGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBEaWN0LCBMaXN0LCBPcHRpb25hbCwgQW55CmZyb20gZGF0'
        'YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcywgZmllbGQKCmltcG9ydCB5YW1sCgp0cnk6CiAgICBpbXBvcnQgeWFtbApleGNl'
        'cHQgSW1wb3J0RXJyb3I6CiAgICByYWlzZSBJbXBvcnRFcnJvcigiUHlZQU1MIG5vdCBpbnN0YWxsZWQuIEluc3RhbGwgd2l0'
        'aDogcGlwIGluc3RhbGwgcHl5YW1sIikKCmltcG9ydCBsb2dnaW5nCgpsb2dnZXIgPSBsb2dnaW5nLmdldExvZ2dlcihfX25h'
        'bWVfXykKCgpAZGF0YWNsYXNzCmNsYXNzIFN0eWxlUHJlc2V0OgogICAgIiIiUmVwcmVzZW50cyBhIG11c2ljYWwgc3R5bGUg'
        'cHJlc2V0LiIiIgogICAgbmFtZTogc3RyCiAgICBiYXNlX3Byb21wdDogc3RyCiAgICBuZWdhdGl2ZV9wcm9tcHQ6IHN0cgog'
        'ICAgdGVtcG9fYnBtOiBpbnQKICAgIGtleTogc3RyCiAgICBpbnN0cnVtZW50YXRpb246IHN0ciA9ICIiCgoKQGRhdGFjbGFz'
        'cwpjbGFzcyBTZWN0aW9uTW9kaWZpZXI6CiAgICAiIiJSZXByZXNlbnRzIG1vZGlmaWVycyBmb3IgYSBzb25nIHNlY3Rpb24g'
        'dHlwZS4iIiIKICAgIG5hbWU6IHN0cgogICAgZW5lcmd5X2xldmVsOiBmbG9hdAogICAgcHJvbXB0X2FkZGl0aW9uczogc3Ry'
        'CiAgICBkdXJhdGlvbl9yYXRpbzogZmxvYXQKCgpAZGF0YWNsYXNzCmNsYXNzIFByZXNldENvbmZpZzoKICAgICIiIkNvbXBs'
        'ZXRlIHByZXNldCBjb25maWd1cmF0aW9uLiIiIgogICAgcHJlc2V0czogRGljdFtzdHIsIFN0eWxlUHJlc2V0XSA9IGZpZWxk'
        'KGRlZmF1bHRfZmFjdG9yeT1kaWN0KQogICAgc2VjdGlvbl9tb2RpZmllcnM6IERpY3Rbc3RyLCBTZWN0aW9uTW9kaWZpZXJd'
        'ID0gZmllbGQoZGVmYXVsdF9mYWN0b3J5PWRpY3QpCgoKY2xhc3MgUHJlc2V0RW5naW5lOgogICAgIiIiCiAgICBFbmdpbmUg'
        'Zm9yIG1hbmFnaW5nIGFuZCBzeW50aGVzaXppbmcgbXVzaWNhbCBwcm9tcHRzIGJhc2VkIG9uIHByZXNldHMuCiAgICAKICAg'
        'IExvYWRzIHN0eWxlIHByZXNldHMgZnJvbSBZQU1MIGNvbmZpZ3VyYXRpb24gYW5kIHByb3ZpZGVzIG1ldGhvZHMKICAgIHRv'
        'IGdlbmVyYXRlIHNlY3Rpb24tc3BlY2lmaWMgcHJvbXB0cyBieSBjb21iaW5pbmcgYmFzZSBzdHlsZXMKICAgIHdpdGggZW5l'
        'cmd5IG1vZGlmaWVycy4KICAgICIiIgogICAgCiAgICBERUZBVUxUX1BSRVNFVF9QQVRIID0gUGF0aChfX2ZpbGVfXykucGFy'
        'ZW50LnBhcmVudCAvICJwcmVzZXRzIiAvICJzdHlsZXMueWFtbCIKICAgIAogICAgZGVmIF9faW5pdF9fKHNlbGYsIHByZXNl'
        'dF9maWxlOiBPcHRpb25hbFtzdHJdID0gTm9uZSk6CiAgICAgICAgIiIiCiAgICAgICAgSW5pdGlhbGl6ZSB0aGUgcHJlc2V0'
        'IGVuZ2luZS4KICAgICAgICAKICAgICAgICBBcmdzOgogICAgICAgICAgICBwcmVzZXRfZmlsZTogUGF0aCB0byBZQU1MIHBy'
        'ZXNldCBmaWxlLiBVc2VzIGRlZmF1bHQgaWYgTm9uZS4KICAgICAgICAiIiIKICAgICAgICBzZWxmLnByZXNldHM6IERpY3Rb'
        'c3RyLCBTdHlsZVByZXNldF0gPSB7fQogICAgICAgIHNlbGYuc2VjdGlvbl9tb2RpZmllcnM6IERpY3Rbc3RyLCBTZWN0aW9u'
        'TW9kaWZpZXJdID0ge30KICAgICAgICAKICAgICAgICBwcmVzZXRfcGF0aCA9IFBhdGgocHJlc2V0X2ZpbGUpIGlmIHByZXNl'
        'dF9maWxlIGVsc2Ugc2VsZi5ERUZBVUxUX1BSRVNFVF9QQVRICiAgICAgICAgCiAgICAgICAgaWYgcHJlc2V0X3BhdGguZXhp'
        'c3RzKCk6CiAgICAgICAgICAgIHNlbGYubG9hZF9wcmVzZXRzKHN0cihwcmVzZXRfcGF0aCkpCiAgICAgICAgZWxzZToKICAg'
        'ICAgICAgICAgbG9nZ2VyLndhcm5pbmcoZiJQcmVzZXQgZmlsZSBub3QgZm91bmQ6IHtwcmVzZXRfcGF0aH0uIFVzaW5nIGVt'
        'cHR5IHByZXNldHMuIikKICAgIAogICAgZGVmIGxvYWRfcHJlc2V0cyhzZWxmLCBwcmVzZXRfZmlsZTogc3RyKSAtPiBOb25l'
        'OgogICAgICAgICIiIgogICAgICAgIExvYWQgcHJlc2V0cyBmcm9tIGEgWUFNTCBmaWxlLgogICAgICAgIAogICAgICAgIEFy'
        'Z3M6CiAgICAgICAgICAgIHByZXNldF9maWxlOiBQYXRoIHRvIFlBTUwgY29uZmlndXJhdGlvbiBmaWxlCiAgICAgICAgIiIi'
        'CiAgICAgICAgdHJ5OgogICAgICAgICAgICB3aXRoIG9wZW4ocHJlc2V0X2ZpbGUsICdyJywgZW5jb2Rpbmc9J3V0Zi04Jykg'
        'YXMgZjoKICAgICAgICAgICAgICAgIGNvbmZpZyA9IHlhbWwuc2FmZV9sb2FkKGYpCiAgICAgICAgICAgIAogICAgICAgICAg'
        'ICBpZiBub3QgY29uZmlnOgogICAgICAgICAgICAgICAgbG9nZ2VyLndhcm5pbmcoZiJFbXB0eSBwcmVzZXQgZmlsZToge3By'
        'ZXNldF9maWxlfSIpCiAgICAgICAgICAgICAgICByZXR1cm4KICAgICAgICAgICAgCiAgICAgICAgICAgICMgTG9hZCBzdHls'
        'ZSBwcmVzZXRzCiAgICAgICAgICAgIHByZXNldHNfZGF0YSA9IGNvbmZpZy5nZXQoJ3ByZXNldHMnLCB7fSkKICAgICAgICAg'
        'ICAgZm9yIG5hbWUsIGRhdGEgaW4gcHJlc2V0c19kYXRhLml0ZW1zKCk6CiAgICAgICAgICAgICAgICBzZWxmLnByZXNldHNb'
        'bmFtZV0gPSBTdHlsZVByZXNldCgKICAgICAgICAgICAgICAgICAgICBuYW1lPW5hbWUsCiAgICAgICAgICAgICAgICAgICAg'
        'YmFzZV9wcm9tcHQ9ZGF0YS5nZXQoJ2Jhc2VfcHJvbXB0JywgJycpLAogICAgICAgICAgICAgICAgICAgIG5lZ2F0aXZlX3By'
        'b21wdD1kYXRhLmdldCgnbmVnYXRpdmVfcHJvbXB0JywgJycpLAogICAgICAgICAgICAgICAgICAgIHRlbXBvX2JwbT1kYXRh'
        'LmdldCgndGVtcG9fYnBtJywgMTIwKSwKICAgICAgICAgICAgICAgICAgICBrZXk9ZGF0YS5nZXQoJ2tleScsICdDIG1ham9y'
        'JyksCiAgICAgICAgICAgICAgICAgICAgaW5zdHJ1bWVudGF0aW9uPWRhdGEuZ2V0KCdpbnN0cnVtZW50YXRpb24nLCAnJykK'
        'ICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgCiAgICAgICAgICAgICMgTG9hZCBzZWN0aW9uIG1vZGlmaWVycwogICAg'
        'ICAgICAgICBtb2RpZmllcnNfZGF0YSA9IGNvbmZpZy5nZXQoJ3NlY3Rpb25fbW9kaWZpZXJzJywge30pCiAgICAgICAgICAg'
        'IGZvciBuYW1lLCBkYXRhIGluIG1vZGlmaWVyc19kYXRhLml0ZW1zKCk6CiAgICAgICAgICAgICAgICBzZWxmLnNlY3Rpb25f'
        'bW9kaWZpZXJzW25hbWVdID0gU2VjdGlvbk1vZGlmaWVyKAogICAgICAgICAgICAgICAgICAgIG5hbWU9bmFtZSwKICAgICAg'
        'ICAgICAgICAgICAgICBlbmVyZ3lfbGV2ZWw9ZGF0YS5nZXQoJ2VuZXJneV9sZXZlbCcsIDAuNyksCiAgICAgICAgICAgICAg'
        'ICAgICAgcHJvbXB0X2FkZGl0aW9ucz1kYXRhLmdldCgncHJvbXB0X2FkZGl0aW9ucycsICcnKSwKICAgICAgICAgICAgICAg'
        'ICAgICBkdXJhdGlvbl9yYXRpbz1kYXRhLmdldCgnZHVyYXRpb25fcmF0aW8nLCAwLjIpCiAgICAgICAgICAgICAgICApCiAg'
        'ICAgICAgICAgIAogICAgICAgICAgICBsb2dnZXIuaW5mbyhmIkxvYWRlZCB7bGVuKHNlbGYucHJlc2V0cyl9IHByZXNldHMg'
        'YW5kIHtsZW4oc2VsZi5zZWN0aW9uX21vZGlmaWVycyl9IHNlY3Rpb24gbW9kaWZpZXJzIikKICAgICAgICAgICAgCiAgICAg'
        'ICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOgogICAgICAgICAgICBsb2dnZXIuZXJyb3IoZiJGYWlsZWQgdG8gbG9hZCBwcmVz'
        'ZXRzIGZyb20ge3ByZXNldF9maWxlfToge2V9IikKICAgICAgICAgICAgcmFpc2UKICAgIAogICAgZGVmIGdldF9wcmVzZXQo'
        'c2VsZiwgcHJlc2V0X25hbWU6IHN0cikgLT4gT3B0aW9uYWxbU3R5bGVQcmVzZXRdOgogICAgICAgICIiIgogICAgICAgIEdl'
        'dCBhIHN0eWxlIHByZXNldCBieSBuYW1lLgogICAgICAgIAogICAgICAgIEFyZ3M6CiAgICAgICAgICAgIHByZXNldF9uYW1l'
        'OiBOYW1lIG9mIHRoZSBwcmVzZXQKICAgICAgICAKICAgICAgICBSZXR1cm5zOgogICAgICAgICAgICBTdHlsZVByZXNldCBp'
        'ZiBmb3VuZCwgTm9uZSBvdGhlcndpc2UKICAgICAgICAiIiIKICAgICAgICBwcmVzZXQgPSBzZWxmLnByZXNldHMuZ2V0KHBy'
        'ZXNldF9uYW1lLmxvd2VyKCkpCiAgICAgICAgaWYgcHJlc2V0IGlzIE5vbmU6CiAgICAgICAgICAgIGxvZ2dlci53YXJuaW5n'
        'KGYiUHJlc2V0ICd7cHJlc2V0X25hbWV9JyBub3QgZm91bmQuIEF2YWlsYWJsZToge2xpc3Qoc2VsZi5wcmVzZXRzLmtleXMo'
        'KSl9IikKICAgICAgICByZXR1cm4gcHJlc2V0CiAgICAKICAgIGRlZiBnZXRfc2VjdGlvbl9tb2RpZmllcihzZWxmLCBzZWN0'
        'aW9uX3R5cGU6IHN0cikgLT4gU2VjdGlvbk1vZGlmaWVyOgogICAgICAgICIiIgogICAgICAgIEdldCBzZWN0aW9uIG1vZGlm'
        'aWVyIGJ5IHR5cGUuCiAgICAgICAgCiAgICAgICAgQXJnczoKICAgICAgICAgICAgc2VjdGlvbl90eXBlOiBUeXBlIG9mIHNl'
        'Y3Rpb24gKGludHJvLCB2ZXJzZSwgY2hvcnVzLCBicmlkZ2UsIG91dHJvKQogICAgICAgIAogICAgICAgIFJldHVybnM6CiAg'
        'ICAgICAgICAgIFNlY3Rpb25Nb2RpZmllciBmb3IgdGhlIHNlY3Rpb24gdHlwZQogICAgICAgICIiIgogICAgICAgIG1vZGlm'
        'aWVyID0gc2VsZi5zZWN0aW9uX21vZGlmaWVycy5nZXQoc2VjdGlvbl90eXBlLmxvd2VyKCkpCiAgICAgICAgaWYgbW9kaWZp'
        'ZXIgaXMgTm9uZToKICAgICAgICAgICAgbG9nZ2VyLndhcm5pbmcoZiJTZWN0aW9uIG1vZGlmaWVyICd7c2VjdGlvbl90eXBl'
        'fScgbm90IGZvdW5kLiBVc2luZyBkZWZhdWx0cy4iKQogICAgICAgICAgICAjIFJldHVybiBhIGRlZmF1bHQgbW9kaWZpZXIK'
        'ICAgICAgICAgICAgcmV0dXJuIFNlY3Rpb25Nb2RpZmllcigKICAgICAgICAgICAgICAgIG5hbWU9c2VjdGlvbl90eXBlLAog'
        'ICAgICAgICAgICAgICAgZW5lcmd5X2xldmVsPTAuNywKICAgICAgICAgICAgICAgIHByb21wdF9hZGRpdGlvbnM9IiIsCiAg'
        'ICAgICAgICAgICAgICBkdXJhdGlvbl9yYXRpbz0wLjIKICAgICAgICAgICAgKQogICAgICAgIHJldHVybiBtb2RpZmllcgog'
        'ICAgCiAgICBkZWYgc3ludGhlc2l6ZV9wcm9tcHQoCiAgICAgICAgc2VsZiwKICAgICAgICBwcmVzZXRfbmFtZTogc3RyLAog'
        'ICAgICAgIHNlY3Rpb25fdHlwZTogc3RyLAogICAgICAgIGN1c3RvbV9seXJpY3NfdGhlbWU6IE9wdGlvbmFsW3N0cl0gPSBO'
        'b25lCiAgICApIC0+IHR1cGxlW3N0ciwgc3RyXToKICAgICAgICAiIiIKICAgICAgICBTeW50aGVzaXplIGEgY29tcGxldGUg'
        'cHJvbXB0IGZvciBhIHNvbmcgc2VjdGlvbi4KICAgICAgICAKICAgICAgICBDb21iaW5lcyB0aGUgYmFzZSBzdHlsZSBwcmVz'
        'ZXQgd2l0aCBzZWN0aW9uLXNwZWNpZmljIGVuZXJneSBtb2RpZmllcnMKICAgICAgICBhbmQgb3B0aW9uYWwgY3VzdG9tIHRo'
        'ZW1lIGVsZW1lbnRzLgogICAgICAgIAogICAgICAgIEFyZ3M6CiAgICAgICAgICAgIHByZXNldF9uYW1lOiBOYW1lIG9mIHRo'
        'ZSBzdHlsZSBwcmVzZXQKICAgICAgICAgICAgc2VjdGlvbl90eXBlOiBUeXBlIG9mIHNlY3Rpb24gKGludHJvLCB2ZXJzZSwg'
        'Y2hvcnVzLCBldGMuKQogICAgICAgICAgICBjdXN0b21fbHlyaWNzX3RoZW1lOiBPcHRpb25hbCBhZGRpdGlvbmFsIHRoZW1l'
        'IGtleXdvcmRzIGZyb20gbHlyaWNzCiAgICAgICAgCiAgICAgICAgUmV0dXJuczoKICAgICAgICAgICAgVHVwbGUgb2YgKHBv'
        'c2l0aXZlX3Byb21wdCwgbmVnYXRpdmVfcHJvbXB0KQogICAgICAgICIiIgogICAgICAgIHByZXNldCA9IHNlbGYuZ2V0X3By'
        'ZXNldChwcmVzZXRfbmFtZSkKICAgICAgICBpZiBwcmVzZXQgaXMgTm9uZToKICAgICAgICAgICAgIyBGYWxsYmFjayB0byBh'
        'IGdlbmVyaWMgcHJvbXB0CiAgICAgICAgICAgIGJhc2VfcHJvbXB0ID0gImluc3RydW1lbnRhbCBtdXNpYywgbWVsb2RpYywg'
        'Y29oZXJlbnQiCiAgICAgICAgICAgIG5lZ2F0aXZlX3Byb21wdCA9ICJub2lzZSwgZGlzdG9ydGlvbiwgc2lsZW5jZSIKICAg'
        'ICAgICBlbHNlOgogICAgICAgICAgICBiYXNlX3Byb21wdCA9IHByZXNldC5iYXNlX3Byb21wdAogICAgICAgICAgICBuZWdh'
        'dGl2ZV9wcm9tcHQgPSBwcmVzZXQubmVnYXRpdmVfcHJvbXB0CiAgICAgICAgCiAgICAgICAgbW9kaWZpZXIgPSBzZWxmLmdl'
        'dF9zZWN0aW9uX21vZGlmaWVyKHNlY3Rpb25fdHlwZSkKICAgICAgICAKICAgICAgICAjIENvbWJpbmUgYmFzZSBwcm9tcHQg'
        'd2l0aCBzZWN0aW9uIG1vZGlmaWVycwogICAgICAgIHBhcnRzID0gW2Jhc2VfcHJvbXB0XQogICAgICAgIAogICAgICAgIGlm'
        'IG1vZGlmaWVyLnByb21wdF9hZGRpdGlvbnM6CiAgICAgICAgICAgIHBhcnRzLmFwcGVuZChtb2RpZmllci5wcm9tcHRfYWRk'
        'aXRpb25zKQogICAgICAgIAogICAgICAgIGlmIGN1c3RvbV9seXJpY3NfdGhlbWU6CiAgICAgICAgICAgIHBhcnRzLmFwcGVu'
        'ZChjdXN0b21fbHlyaWNzX3RoZW1lKQogICAgICAgIAogICAgICAgICMgQWRkIGluc3RydW1lbnRhdGlvbiBpZiBhdmFpbGFi'
        'bGUKICAgICAgICBpZiBwcmVzZXQgYW5kIHByZXNldC5pbnN0cnVtZW50YXRpb246CiAgICAgICAgICAgIHBhcnRzLmFwcGVu'
        'ZChmImZlYXR1cmluZyB7cHJlc2V0Lmluc3RydW1lbnRhdGlvbn0iKQogICAgICAgIAogICAgICAgIHBvc2l0aXZlX3Byb21w'
        'dCA9ICIsICIuam9pbihwYXJ0cykKICAgICAgICAKICAgICAgICBsb2dnZXIuZGVidWcoZiJTeW50aGVzaXplZCBwcm9tcHQg'
        'Zm9yIHtwcmVzZXRfbmFtZX0ve3NlY3Rpb25fdHlwZX06IHtwb3NpdGl2ZV9wcm9tcHRbOjgwXX0uLi4iKQogICAgICAgIAog'
        'ICAgICAgIHJldHVybiBwb3NpdGl2ZV9wcm9tcHQsIG5lZ2F0aXZlX3Byb21wdAogICAgCiAgICBkZWYgbGlzdF9wcmVzZXRz'
        'KHNlbGYpIC0+IExpc3Rbc3RyXToKICAgICAgICAiIiJHZXQgbGlzdCBvZiBhdmFpbGFibGUgcHJlc2V0IG5hbWVzLiIiIgog'
        'ICAgICAgIHJldHVybiBsaXN0KHNlbGYucHJlc2V0cy5rZXlzKCkpCiAgICAKICAgIGRlZiBsaXN0X3NlY3Rpb25fdHlwZXMo'
        'c2VsZikgLT4gTGlzdFtzdHJdOgogICAgICAgICIiIkdldCBsaXN0IG9mIGF2YWlsYWJsZSBzZWN0aW9uIHR5cGVzLiIiIgog'
        'ICAgICAgIHJldHVybiBsaXN0KHNlbGYuc2VjdGlvbl9tb2RpZmllcnMua2V5cygpKQogICAgCiAgICBkZWYgY2FsY3VsYXRl'
        'X3NlY3Rpb25fZHVyYXRpb25zKAogICAgICAgIHNlbGYsCiAgICAgICAgdGFyZ2V0X2R1cmF0aW9uX3NlY29uZHM6IGZsb2F0'
        'LAogICAgICAgIHNlY3Rpb25zOiBMaXN0W3N0cl0KICAgICkgLT4gRGljdFtzdHIsIGZsb2F0XToKICAgICAgICAiIiIKICAg'
        'ICAgICBDYWxjdWxhdGUgZHVyYXRpb24gZm9yIGVhY2ggc2VjdGlvbiBiYXNlZCBvbiB0YXJnZXQgdG90YWwgZHVyYXRpb24u'
        'CiAgICAgICAgCiAgICAgICAgQXJnczoKICAgICAgICAgICAgdGFyZ2V0X2R1cmF0aW9uX3NlY29uZHM6IFRhcmdldCB0b3Rh'
        'bCBzb25nIGR1cmF0aW9uCiAgICAgICAgICAgIHNlY3Rpb25zOiBMaXN0IG9mIHNlY3Rpb24gdHlwZXMgaW4gb3JkZXIKICAg'
        'ICAgICAKICAgICAgICBSZXR1cm5zOgogICAgICAgICAgICBEaWN0aW9uYXJ5IG1hcHBpbmcgc2VjdGlvbiBpbmRleCB0byBk'
        'dXJhdGlvbiBpbiBzZWNvbmRzCiAgICAgICAgIiIiCiAgICAgICAgZHVyYXRpb25zID0ge30KICAgICAgICAKICAgICAgICAj'
        'IEdldCBkdXJhdGlvbiByYXRpb3MgZm9yIGVhY2ggc2VjdGlvbgogICAgICAgIHRvdGFsX3JhdGlvID0gMC4wCiAgICAgICAg'
        'cmF0aW9zID0gW10KICAgICAgICBmb3Igc2VjdGlvbiBpbiBzZWN0aW9uczoKICAgICAgICAgICAgbW9kaWZpZXIgPSBzZWxm'
        'LmdldF9zZWN0aW9uX21vZGlmaWVyKHNlY3Rpb24pCiAgICAgICAgICAgIHJhdGlvcy5hcHBlbmQobW9kaWZpZXIuZHVyYXRp'
        'b25fcmF0aW8pCiAgICAgICAgICAgIHRvdGFsX3JhdGlvICs9IG1vZGlmaWVyLmR1cmF0aW9uX3JhdGlvCiAgICAgICAgCiAg'
        'ICAgICAgIyBOb3JtYWxpemUgcmF0aW9zIGlmIHRoZXkgZG9uJ3Qgc3VtIHRvIDEuMAogICAgICAgIGlmIHRvdGFsX3JhdGlv'
        'ID4gMDoKICAgICAgICAgICAgbm9ybWFsaXplZF9yYXRpb3MgPSBbciAvIHRvdGFsX3JhdGlvIGZvciByIGluIHJhdGlvc10K'
        'ICAgICAgICBlbHNlOgogICAgICAgICAgICAjIEVxdWFsIGRpc3RyaWJ1dGlvbiBpZiBubyByYXRpb3MgZGVmaW5lZAogICAg'
        'ICAgICAgICBlcXVhbF9yYXRpbyA9IDEuMCAvIGxlbihzZWN0aW9ucykgaWYgc2VjdGlvbnMgZWxzZSAwCiAgICAgICAgICAg'
        'IG5vcm1hbGl6ZWRfcmF0aW9zID0gW2VxdWFsX3JhdGlvXSAqIGxlbihzZWN0aW9ucykKICAgICAgICAKICAgICAgICAjIENh'
        'bGN1bGF0ZSBhY3R1YWwgZHVyYXRpb25zCiAgICAgICAgZm9yIGksIHJhdGlvIGluIGVudW1lcmF0ZShub3JtYWxpemVkX3Jh'
        'dGlvcyk6CiAgICAgICAgICAgIGR1cmF0aW9uc1tpXSA9IHRhcmdldF9kdXJhdGlvbl9zZWNvbmRzICogcmF0aW8KICAgICAg'
        'ICAKICAgICAgICByZXR1cm4gZHVyYXRpb25zCiAgICAKICAgIGRlZiBnZXRfZW5lcmd5X2FkanVzdGVkX3BhcmFtcygKICAg'
        'ICAgICBzZWxmLAogICAgICAgIHByZXNldF9uYW1lOiBzdHIsCiAgICAgICAgc2VjdGlvbl90eXBlOiBzdHIKICAgICkgLT4g'
        'RGljdFtzdHIsIEFueV06CiAgICAgICAgIiIiCiAgICAgICAgR2V0IGdlbmVyYXRpb24gcGFyYW1ldGVycyBhZGp1c3RlZCBm'
        'b3Igc2VjdGlvbiBlbmVyZ3kgbGV2ZWwuCiAgICAgICAgCiAgICAgICAgQXJnczoKICAgICAgICAgICAgcHJlc2V0X25hbWU6'
        'IFN0eWxlIHByZXNldCBuYW1lCiAgICAgICAgICAgIHNlY3Rpb25fdHlwZTogU2VjdGlvbiB0eXBlCiAgICAgICAgCiAgICAg'
        'ICAgUmV0dXJuczoKICAgICAgICAgICAgRGljdGlvbmFyeSB3aXRoIGFkanVzdGVkIGdlbmVyYXRpb24gcGFyYW1ldGVycwog'
        'ICAgICAgICIiIgogICAgICAgIHByZXNldCA9IHNlbGYuZ2V0X3ByZXNldChwcmVzZXRfbmFtZSkKICAgICAgICBtb2RpZmll'
        'ciA9IHNlbGYuZ2V0X3NlY3Rpb25fbW9kaWZpZXIoc2VjdGlvbl90eXBlKQogICAgICAgIAogICAgICAgICMgQWRqdXN0IGd1'
        'aWRhbmNlIHNjYWxlIGJhc2VkIG9uIGVuZXJneQogICAgICAgICMgSGlnaGVyIGVuZXJneSA9IGxvd2VyIGd1aWRhbmNlICht'
        'b3JlIGNyZWF0aXZlL2ZyZW56aWVkKQogICAgICAgIGJhc2VfZ3VpZGFuY2UgPSA3LjAKICAgICAgICBlbmVyZ3lfYWRqdXN0'
        'bWVudCA9ICgxLjAgLSBtb2RpZmllci5lbmVyZ3lfbGV2ZWwpICogMi4wCiAgICAgICAgZ3VpZGFuY2Vfc2NhbGUgPSBiYXNl'
        'X2d1aWRhbmNlICsgZW5lcmd5X2FkanVzdG1lbnQKICAgICAgICAKICAgICAgICAjIEFkanVzdCBzdGVwcyBiYXNlZCBvbiBl'
        'bmVyZ3kKICAgICAgICAjIEhpZ2hlciBlbmVyZ3kgbWlnaHQgYmVuZWZpdCBmcm9tIG1vcmUgc3RlcHMgZm9yIGNsYXJpdHkK'
        'ICAgICAgICBiYXNlX3N0ZXBzID0gNTAKICAgICAgICBzdGVwcyA9IGludChiYXNlX3N0ZXBzICsgKG1vZGlmaWVyLmVuZXJn'
        'eV9sZXZlbCAqIDEwKSkKICAgICAgICAKICAgICAgICByZXR1cm4gewogICAgICAgICAgICAiZ3VpZGFuY2Vfc2NhbGUiOiBy'
        'b3VuZChndWlkYW5jZV9zY2FsZSwgMSksCiAgICAgICAgICAgICJudW1faW5mZXJlbmNlX3N0ZXBzIjogbWluKHN0ZXBzLCAx'
        'MDApLAogICAgICAgICAgICAiZW5lcmd5X2xldmVsIjogbW9kaWZpZXIuZW5lcmd5X2xldmVsCiAgICAgICAgfQo='
    ),
    'src/song_arranger.py': (
        '...'
    ),
    'src/tunnel_manager.py': (
        'IiIiClR1bm5lbCBNYW5hZ2VyIGZvciBSaWZmdXNpb24gS2FnZ2xlIFNvbmcgU3R1ZGlvCgpNYW5hZ2VzIGNsb3VkZmxhcmVk'
        'IHR1bm5lbCBjcmVhdGlvbiBmb3IgZXhwb3NpbmcgdGhlIEZhc3RBUEkgc2VydmVyCnRvIHRoZSBwdWJsaWMgaW50ZXJuZXQg'
        'ZnJvbSB3aXRoaW4gYSBLYWdnbGUgbm90ZWJvb2suCiIiIgoKaW1wb3J0IG9zCmltcG9ydCBzdWJwcm9jZXNzCmltcG9ydCB0'
        'aW1lCmltcG9ydCByZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0eXBpbmcgaW1wb3J0IE9wdGlvbmFsLCBUdXBs'
        'ZQppbXBvcnQgbG9nZ2luZwppbXBvcnQgdGhyZWFkaW5nCgpsb2dnZXIgPSBsb2dnaW5nLmdldExvZ2dlcihfX25hbWVfXykK'
        'CgpjbGFzcyBUdW5uZWxNYW5hZ2VyOgogICAgIiIiCiAgICBNYW5hZ2VzIGNsb3VkZmxhcmVkIHR1bm5lbCBmb3IgZXhwb3Np'
        'bmcgbG9jYWwgc2VydmVyLgogICAgCiAgICBDcmVhdGVzIGEgc2VjdXJlIHR1bm5lbCBmcm9tIHRoZSBLYWdnbGUgbm90ZWJv'
        'b2sgdG8gY2xvdWRmbGFyZSwKICAgIHByb3ZpZGluZyBhIHB1YmxpYyBVUkwgZm9yIHRoZSBBUEkgc2VydmVyLgogICAgIiIi'
        'CiAgICAKICAgIGRlZiBfX2luaXRfXyhzZWxmKToKICAgICAgICAiIiJJbml0aWFsaXplIHR1bm5lbCBtYW5hZ2VyLiIiIgog'
        'ICAgICAgIHNlbGYudHVubmVsX3Byb2Nlc3M6IE9wdGlvbmFsW3N1YnByb2Nlc3MuUG9wZW5dID0gTm9uZQogICAgICAgIHNl'
        'bGYudHVubmVsX3VybDogT3B0aW9uYWxbc3RyXSA9IE5vbmUKICAgICAgICBzZWxmLmlzX3J1bm5pbmcgPSBGYWxzZQogICAg'
        'ICAgIHNlbGYuX2xvZ190aHJlYWQ6IE9wdGlvbmFsW3RocmVhZGluZy5UaHJlYWRdID0gTm9uZQogICAgCiAgICBkZWYgaW5z'
        'dGFsbF9jbG91ZGZsYXJlZChzZWxmKSAtPiBib29sOgogICAgICAgICIiIgogICAgICAgIEluc3RhbGwgY2xvdWRmbGFyZWQg'
        'aW4gdGhlIEthZ2dsZSBlbnZpcm9ubWVudC4KICAgICAgICAKICAgICAgICBSZXR1cm5zOgogICAgICAgICAgICBUcnVlIGlm'
        'IGluc3RhbGxhdGlvbiBzdWNjZXNzZnVsCiAgICAgICAgIiIiCiAgICAgICAgdHJ5OgogICAgICAgICAgICAjIENoZWNrIGlm'
        'IGFscmVhZHkgaW5zdGFsbGVkCiAgICAgICAgICAgIHJlc3VsdCA9IHN1YnByb2Nlc3MucnVuKAogICAgICAgICAgICAgICAg'
        'WyJ3aGljaCIsICJjbG91ZGZsYXJlZCJdLAogICAgICAgICAgICAgICAgY2FwdHVyZV9vdXRwdXQ9VHJ1ZSwKICAgICAgICAg'
        'ICAgICAgIHRleHQ9VHJ1ZQogICAgICAgICAgICApCiAgICAgICAgICAgIAogICAgICAgICAgICBpZiByZXN1bHQucmV0dXJu'
        'Y29kZSA9PSAwOgogICAgICAgICAgICAgICAgbG9nZ2VyLmluZm8oImNsb3VkZmxhcmVkIGFscmVhZHkgaW5zdGFsbGVkIikK'
        'ICAgICAgICAgICAgICAgIHJldHVybiBUcnVlCiAgICAgICAgICAgIAogICAgICAgICAgICAjIERvd25sb2FkIGFuZCBpbnN0'
        'YWxsIGNsb3VkZmxhcmVkCiAgICAgICAgICAgIGxvZ2dlci5pbmZvKCJJbnN0YWxsaW5nIGNsb3VkZmxhcmVkLi4uIikKICAg'
        'ICAgICAgICAgCiAgICAgICAgICAgICMgRG93bmxvYWQgbGF0ZXN0IHZlcnNpb24KICAgICAgICAgICAgZG93bmxvYWRfY21k'
        'ID0gKAogICAgICAgICAgICAgICAgImN1cmwgLUwgLS1vdXRwdXQgL3RtcC9jbG91ZGZsYXJlZC5kZWIgIgogICAgICAgICAg'
        'ICAgICAgImh0dHBzOi8vZ2l0aHViLmNvbS9jbG91ZGZsYXJlL2Nsb3VkZmxhcmVkL3JlbGVhc2VzL2xhdGVzdC9kb3dubG9h'
        'ZC8iCiAgICAgICAgICAgICAgICAiY2xvdWRmbGFyZWQtbGludXgtYW1kNjQuZGViIgogICAgICAgICAgICApCiAgICAgICAg'
        'ICAgIAogICAgICAgICAgICByZXN1bHQgPSBzdWJwcm9jZXNzLnJ1bigKICAgICAgICAgICAgICAgIGRvd25sb2FkX2NtZCwK'
        'ICAgICAgICAgICAgICAgIHNoZWxsPVRydWUsCiAgICAgICAgICAgICAgICBjYXB0dXJlX291dHB1dD1UcnVlLAogICAgICAg'
        'ICAgICAgICAgdGV4dD1UcnVlCiAgICAgICAgICAgICkKICAgICAgICAgICAgCiAgICAgICAgICAgIGlmIHJlc3VsdC5yZXR1'
        'cm5jb2RlICE9IDA6CiAgICAgICAgICAgICAgICBsb2dnZXIuZXJyb3IoZiJEb3dubG9hZCBmYWlsZWQ6IHtyZXN1bHQuc3Rk'
        'ZXJyfSIpCiAgICAgICAgICAgICAgICByZXR1cm4gRmFsc2UKICAgICAgICAgICAgCiAgICAgICAgICAgICMgSW5zdGFsbCBk'
        'ZWIgcGFja2FnZQogICAgICAgICAgICBpbnN0YWxsX2NtZCA9ICJkcGtnIC1pIC90bXAvY2xvdWRmbGFyZWQuZGViIgogICAg'
        'ICAgICAgICByZXN1bHQgPSBzdWJwcm9jZXNzLnJ1bigKICAgICAgICAgICAgICAgIGluc3RhbGxfY21kLAogICAgICAgICAg'
        'ICAgICAgc2hlbGw9VHJ1ZSwKICAgICAgICAgICAgICAgIGNhcHR1cmVfb3V0cHV0PVRydWUsCiAgICAgICAgICAgICAgICB0'
        'ZXh0PVRydWUKICAgICAgICAgICAgKQogICAgICAgICAgICAKICAgICAgICAgICAgaWYgcmVzdWx0LnJldHVybmNvZGUgIT0g'
        'MDoKICAgICAgICAgICAgICAgIGxvZ2dlci5lcnJvcihmIkluc3RhbGxhdGlvbiBmYWlsZWQ6IHtyZXN1bHQuc3RkZXJyfSIp'
        'CiAgICAgICAgICAgICAgICByZXR1cm4gRmFsc2UKICAgICAgICAgICAgCiAgICAgICAgICAgIGxvZ2dlci5pbmZvKCJjbG91'
        'ZGZsYXJlZCBpbnN0YWxsZWQgc3VjY2Vzc2Z1bGx5IikKICAgICAgICAgICAgcmV0dXJuIFRydWUKICAgICAgICAgICAgCiAg'
        'ICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOgogICAgICAgICAgICBsb2dnZXIuZXJyb3IoZiJJbnN0YWxsYXRpb24gZXJy'
        'b3I6IHtlfSIpCiAgICAgICAgICAgIHJldHVybiBGYWxzZQogICAgCiAgICBkZWYgc3RhcnRfdHVubmVsKHNlbGYsIHBvcnQ6'
        'IGludCA9IDgwMDApIC0+IFR1cGxlW2Jvb2wsIE9wdGlvbmFsW3N0cl1dOgogICAgICAgICIiIgogICAgICAgIFN0YXJ0IGEg'
        'Y2xvdWRmbGFyZWQgdHVubmVsIHRvIGV4cG9zZSB0aGUgbG9jYWwgc2VydmVyLgogICAgICAgIAogICAgICAgIEFyZ3M6CiAg'
        'ICAgICAgICAgIHBvcnQ6IExvY2FsIHBvcnQgdG8gZXhwb3NlCiAgICAgICAgCiAgICAgICAgUmV0dXJuczoKICAgICAgICAg'
        'ICAgVHVwbGUgb2YgKHN1Y2Nlc3MsIHB1YmxpY191cmwpCiAgICAgICAgIiIiCiAgICAgICAgaWYgc2VsZi5pc19ydW5uaW5n'
        'OgogICAgICAgICAgICBsb2dnZXIud2FybmluZygiVHVubmVsIGFscmVhZHkgcnVubmluZyIpCiAgICAgICAgICAgIHJldHVy'
        'biBUcnVlLCBzZWxmLnR1bm5lbF91cmwKICAgICAgICAKICAgICAgICAjIEVuc3VyZSBjbG91ZGZsYXJlZCBpcyBpbnN0YWxs'
        'ZWQKICAgICAgICBpZiBub3Qgc2VsZi5pbnN0YWxsX2Nsb3VkZmxhcmVkKCk6CiAgICAgICAgICAgIHJldHVybiBGYWxzZSwg'
        'Tm9uZQogICAgICAgIAogICAgICAgIHRyeToKICAgICAgICAgICAgIyBTdGFydCBjbG91ZGZsYXJlZCB0dW5uZWwKICAgICAg'
        'ICAgICAgIyBVc2luZyBxdWljayB0dW5uZWwgbW9kZSAobm8gYWNjb3VudCByZXF1aXJlZCkKICAgICAgICAgICAgY21kID0g'
        'WwogICAgICAgICAgICAgICAgImNsb3VkZmxhcmVkIiwKICAgICAgICAgICAgICAgICJ0dW5uZWwiLAogICAgICAgICAgICAg'
        'ICAgIi0tdXJsIiwKICAgICAgICAgICAgICAgIGYiaHR0cDovL2xvY2FsaG9zdDp7cG9ydH0iCiAgICAgICAgICAgIF0KICAg'
        'ICAgICAgICAgCiAgICAgICAgICAgIGxvZ2dlci5pbmZvKGYiU3RhcnRpbmcgdHVubmVsOiB7JyAnLmpvaW4oY21kKX0iKQog'
        'ICAgICAgICAgICAKICAgICAgICAgICAgc2VsZi50dW5uZWxfcHJvY2VzcyA9IHN1YnByb2Nlc3MuUG9wZW4oCiAgICAgICAg'
        'ICAgICAgICBjbWQsCiAgICAgICAgICAgICAgICBzdGRvdXQ9c3VicHJvY2Vzcy5QSVBFLAogICAgICAgICAgICAgICAgc3Rk'
        'ZXJyPXN1YnByb2Nlc3MuU1RET1VULAogICAgICAgICAgICAgICAgdGV4dD1UcnVlLAogICAgICAgICAgICAgICAgYnVmc2l6'
        'ZT0xCiAgICAgICAgICAgICkKICAgICAgICAgICAgCiAgICAgICAgICAgICMgU3RhcnQgbG9nIHJlYWRlciB0aHJlYWQKICAg'
        'ICAgICAgICAgc2VsZi5fbG9nX3RocmVhZCA9IHRocmVhZGluZy5UaHJlYWQoCiAgICAgICAgICAgICAgICB0YXJnZXQ9c2Vs'
        'Zi5fcmVhZF90dW5uZWxfbG9ncywKICAgICAgICAgICAgICAgIGRhZW1vbj1UcnVlCiAgICAgICAgICAgICkKICAgICAgICAg'
        'ICAgc2VsZi5fbG9nX3RocmVhZC5zdGFydCgpCiAgICAgICAgICAgIAogICAgICAgICAgICAjIFdhaXQgZm9yIHR1bm5lbCBV'
        'UkwgdG8gYXBwZWFyCiAgICAgICAgICAgIG1heF93YWl0ID0gMzAgICMgc2Vjb25kcwogICAgICAgICAgICBzdGFydF90aW1l'
        'ID0gdGltZS50aW1lKCkKICAgICAgICAgICAgCiAgICAgICAgICAgIHdoaWxlIHRpbWUudGltZSgpIC0gc3RhcnRfdGltZSA8'
        'IG1heF93YWl0OgogICAgICAgICAgICAgICAgaWYgc2VsZi50dW5uZWxfdXJsOgogICAgICAgICAgICAgICAgICAgIHNlbGYu'
        'aXNfcnVubmluZyA9IFRydWUKICAgICAgICAgICAgICAgICAgICBsb2dnZXIuaW5mbyhmIlR1bm5lbCBzdGFydGVkOiB7c2Vs'
        'Zi50dW5uZWxfdXJsfSIpCiAgICAgICAgICAgICAgICAgICAgcmV0dXJuIFRydWUsIHNlbGYudHVubmVsX3VybAogICAgICAg'
        'ICAgICAgICAgCiAgICAgICAgICAgICAgICAjIENoZWNrIGlmIHByb2Nlc3MgZGllZAogICAgICAgICAgICAgICAgaWYgc2Vs'
        'Zi50dW5uZWxfcHJvY2Vzcy5wb2xsKCkgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICAgICAgbG9nZ2VyLmVycm9yKCJU'
        'dW5uZWwgcHJvY2VzcyBleGl0ZWQgdW5leHBlY3RlZGx5IikKICAgICAgICAgICAgICAgICAgICByZXR1cm4gRmFsc2UsIE5v'
        'bmUKICAgICAgICAgICAgICAgIAogICAgICAgICAgICAgICAgdGltZS5zbGVlcCgxKQogICAgICAgICAgICAKICAgICAgICAg'
        'ICAgbG9nZ2VyLmVycm9yKCJUaW1lb3V0IHdhaXRpbmcgZm9yIHR1bm5lbCBVUkwiKQogICAgICAgICAgICByZXR1cm4gRmFs'
        'c2UsIE5vbmUKICAgICAgICAgICAgCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOgogICAgICAgICAgICBsb2dnZXIu'
        'ZXJyb3IoZiJUdW5uZWwgc3RhcnQgZXJyb3I6IHtlfSIpCiAgICAgICAgICAgIHJldHVybiBGYWxzZSwgTm9uZQogICAgCiAg'
        'ICBkZWYgX3JlYWRfdHVubmVsX2xvZ3Moc2VsZikgLT4gTm9uZToKICAgICAgICAiIiJSZWFkIGFuZCBwYXJzZSB0dW5uZWwg'
        'bG9ncyB0byBleHRyYWN0IHB1YmxpYyBVUkwuIiIiCiAgICAgICAgaWYgbm90IHNlbGYudHVubmVsX3Byb2Nlc3Mgb3Igbm90'
        'IHNlbGYudHVubmVsX3Byb2Nlc3Muc3Rkb3V0OgogICAgICAgICAgICByZXR1cm4KICAgICAgICAKICAgICAgICAjIFBhdHRl'
        'cm4gdG8gbWF0Y2ggdHVubmVsIFVSTAogICAgICAgIHVybF9wYXR0ZXJuID0gcmUuY29tcGlsZSgKICAgICAgICAgICAgcido'
        'dHRwczovL1thLXpBLVowLTktXStcLnRyeWNsb3VkZmxhcmVcLmNvbScKICAgICAgICApCiAgICAgICAgCiAgICAgICAgZm9y'
        'IGxpbmUgaW4gaXRlcihzZWxmLnR1bm5lbF9wcm9jZXNzLnN0ZG91dC5yZWFkbGluZSwgJycpOgogICAgICAgICAgICBpZiBs'
        'aW5lOgogICAgICAgICAgICAgICAgbG9nZ2VyLmRlYnVnKGYiY2xvdWRmbGFyZWQ6IHtsaW5lLnN0cmlwKCl9IikKICAgICAg'
        'ICAgICAgICAgIAogICAgICAgICAgICAgICAgIyBMb29rIGZvciBVUkwgaW4gbG9ncwogICAgICAgICAgICAgICAgbWF0Y2gg'
        'PSB1cmxfcGF0dGVybi5zZWFyY2gobGluZSkKICAgICAgICAgICAgICAgIGlmIG1hdGNoIGFuZCBub3Qgc2VsZi50dW5uZWxf'
        'dXJsOgogICAgICAgICAgICAgICAgICAgIHNlbGYudHVubmVsX3VybCA9IG1hdGNoLmdyb3VwKDApCiAgICAgICAgICAgICAg'
        'ICAgICAgbG9nZ2VyLmluZm8oZiJGb3VuZCB0dW5uZWwgVVJMOiB7c2VsZi50dW5uZWxfdXJsfSIpCiAgICAKICAgIGRlZiBz'
        'dG9wX3R1bm5lbChzZWxmKSAtPiBOb25lOgogICAgICAgICIiIlN0b3AgdGhlIHR1bm5lbC4iIiIKICAgICAgICBpZiBub3Qg'
        'c2VsZi5pc19ydW5uaW5nOgogICAgICAgICAgICByZXR1cm4KICAgICAgICAKICAgICAgICBsb2dnZXIuaW5mbygiU3RvcHBp'
        'bmcgdHVubmVsLi4uIikKICAgICAgICAKICAgICAgICBpZiBzZWxmLnR1bm5lbF9wcm9jZXNzOgogICAgICAgICAgICB0cnk6'
        'CiAgICAgICAgICAgICAgICBzZWxmLnR1bm5lbF9wcm9jZXNzLnRlcm1pbmF0ZSgpCiAgICAgICAgICAgICAgICBzZWxmLnR1'
        'bm5lbF9wcm9jZXNzLndhaXQodGltZW91dD01KQogICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAg'
        'ICAgICAgICBsb2dnZXIuZXJyb3IoZiJFcnJvciBzdG9wcGluZyB0dW5uZWw6IHtlfSIpCiAgICAgICAgICAgICAgICBpZiBz'
        'ZWxmLnR1bm5lbF9wcm9jZXNzOgogICAgICAgICAgICAgICAgICAgIHNlbGYudHVubmVsX3Byb2Nlc3Mua2lsbCgpCiAgICAg'
        'ICAgCiAgICAgICAgc2VsZi50dW5uZWxfcHJvY2VzcyA9IE5vbmUKICAgICAgICBzZWxmLnR1bm5lbF91cmwgPSBOb25lCiAg'
        'ICAgICAgc2VsZi5pc19ydW5uaW5nID0gRmFsc2UKICAgIAogICAgZGVmIGdldF90dW5uZWxfdXJsKHNlbGYpIC0+IE9wdGlv'
        'bmFsW3N0cl06CiAgICAgICAgIiIiR2V0IHRoZSBjdXJyZW50IHR1bm5lbCBVUkwuIiIiCiAgICAgICAgcmV0dXJuIHNlbGYu'
        'dHVubmVsX3VybAogICAgCiAgICBkZWYgaXNfdHVubmVsX3J1bm5pbmcoc2VsZikgLT4gYm9vbDoKICAgICAgICAiIiJDaGVj'
        'ayBpZiB0dW5uZWwgaXMgY3VycmVudGx5IHJ1bm5pbmcuIiIiCiAgICAgICAgcmV0dXJuIHNlbGYuaXNfcnVubmluZyBhbmQg'
        'c2VsZi50dW5uZWxfdXJsIGlzIG5vdCBOb25lCiAgICAKICAgIGRlZiBnZXRfY29ubmVjdGlvbl9pbmZvKHNlbGYpIC0+IGRp'
        'Y3Q6CiAgICAgICAgIiIiR2V0IGNvbXBsZXRlIGNvbm5lY3Rpb24gaW5mb3JtYXRpb24gZm9yIGNsaWVudHMuIiIiCiAgICAg'
        'ICAgcmV0dXJuIHsKICAgICAgICAgICAgInR1bm5lbF91cmwiOiBzZWxmLnR1bm5lbF91cmwsCiAgICAgICAgICAgICJpc19y'
        'dW5uaW5nIjogc2VsZi5pc19ydW5uaW5nLAogICAgICAgICAgICAibG9jYWxfcG9ydCI6IDgwMDAgaWYgc2VsZi5pc19ydW5u'
        'aW5nIGVsc2UgTm9uZQogICAgICAgIH0KCgpkZWYgY3JlYXRlX3F1aWNrX3R1bm5lbChwb3J0OiBpbnQgPSA4MDAwKSAtPiBP'
        'cHRpb25hbFtzdHJdOgogICAgIiIiCiAgICBDb252ZW5pZW5jZSBmdW5jdGlvbiB0byBjcmVhdGUgYSBxdWljayB0dW5uZWwu'
        'CiAgICAKICAgIEFyZ3M6CiAgICAgICAgcG9ydDogTG9jYWwgcG9ydCB0byBleHBvc2UKICAgIAogICAgUmV0dXJuczoKICAg'
        'ICAgICBQdWJsaWMgdHVubmVsIFVSTCBvciBOb25lIGlmIGZhaWxlZAogICAgIiIiCiAgICBtYW5hZ2VyID0gVHVubmVsTWFu'
        'YWdlcigpCiAgICBzdWNjZXNzLCB1cmwgPSBtYW5hZ2VyLnN0YXJ0X3R1bm5lbChwb3J0KQogICAgCiAgICBpZiBzdWNjZXNz'
        'OgogICAgICAgIHJldHVybiB1cmwKICAgIHJldHVybiBOb25lCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgICMg'
        'VGVzdCB0dW5uZWwgY3JlYXRpb24KICAgIHByaW50KCJUZXN0aW5nIGNsb3VkZmxhcmVkIHR1bm5lbC4uLiIpCiAgICAKICAg'
        'IG1hbmFnZXIgPSBUdW5uZWxNYW5hZ2VyKCkKICAgIHN1Y2Nlc3MsIHVybCA9IG1hbmFnZXIuc3RhcnRfdHVubmVsKDgwMDAp'
        'CiAgICAKICAgIGlmIHN1Y2Nlc3MgYW5kIHVybDoKICAgICAgICBwcmludChmIlxu4pyFIFR1bm5lbCBjcmVhdGVkIHN1Y2Nl'
        'c3NmdWxseSEiKQogICAgICAgIHByaW50KGYiUHVibGljIFVSTDoge3VybH0iKQogICAgICAgIHByaW50KGYiXG5LZWVwIHRo'
        'aXMgc2NyaXB0IHJ1bm5pbmcgdG8gbWFpbnRhaW4gdGhlIHR1bm5lbC4iKQogICAgICAgIHByaW50KCJQcmVzcyBDdHJsK0Mg'
        'dG8gc3RvcC5cbiIpCiAgICAgICAgCiAgICAgICAgdHJ5OgogICAgICAgICAgICB3aGlsZSBUcnVlOgogICAgICAgICAgICAg'
        'ICAgdGltZS5zbGVlcCgxKQogICAgICAgIGV4Y2VwdCBLZXlib2FyZEludGVycnVwdDoKICAgICAgICAgICAgcHJpbnQoIlxu'
        'U3RvcHBpbmcgdHVubmVsLi4uIikKICAgICAgICAgICAgbWFuYWdlci5zdG9wX3R1bm5lbCgpCiAgICAgICAgICAgIHByaW50'
        'KCJUdW5uZWwgc3RvcHBlZC4iKQogICAgZWxzZToKICAgICAgICBwcmludCgi4p2MIEZhaWxlZCB0byBjcmVhdGUgdHVubmVs'
        'IikK'
    ),
    'src/utils.py': (
        'IiIiClV0aWxpdHkgZnVuY3Rpb25zIGZvciBSaWZmdXNpb24gS2FnZ2xlIFNvbmcgU3R1ZGlvCiIiIgoKaW1wb3J0IG9zCmlt'
        'cG9ydCBsb2dnaW5nCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgT3B0aW9uYWwsIExpc3Qs'
        'IERpY3QsIEFueQppbXBvcnQgaGFzaGxpYgoKbG9nZ2VyID0gbG9nZ2luZy5nZXRMb2dnZXIoX19uYW1lX18pCgoKZGVmIHNl'
        'dHVwX2xvZ2dpbmcobGV2ZWw6IGludCA9IGxvZ2dpbmcuSU5GTykgLT4gTm9uZToKICAgICIiIgogICAgQ29uZmlndXJlIGxv'
        'Z2dpbmcgZm9yIHRoZSBhcHBsaWNhdGlvbi4KICAgIAogICAgQXJnczoKICAgICAgICBsZXZlbDogTG9nZ2luZyBsZXZlbCAo'
        'ZGVmYXVsdCBJTkZPKQogICAgIiIiCiAgICBsb2dnaW5nLmJhc2ljQ29uZmlnKAogICAgICAgIGxldmVsPWxldmVsLAogICAg'
        'ICAgIGZvcm1hdD0nJShhc2N0aW1lKXMgLSAlKG5hbWUpcyAtICUobGV2ZWxuYW1lKXMgLSAlKG1lc3NhZ2UpcycsCiAgICAg'
        'ICAgZGF0ZWZtdD0nJVktJW0tJWQgJUg6JU06JVMnCiAgICApCgoKZGVmIGVuc3VyZV9kaXJlY3RvcnkocGF0aDogc3RyLCBj'
        'cmVhdGVfcGFyZW50czogYm9vbCA9IFRydWUpIC0+IFBhdGg6CiAgICAiIiIKICAgIEVuc3VyZSBhIGRpcmVjdG9yeSBleGlz'
        'dHMsIGNyZWF0aW5nIGl0IGlmIG5lY2Vzc2FyeS4KICAgIAogICAgQXJnczoKICAgICAgICBwYXRoOiBEaXJlY3RvcnkgcGF0'
        'aAogICAgICAgIGNyZWF0ZV9wYXJlbnRzOiBXaGV0aGVyIHRvIGNyZWF0ZSBwYXJlbnQgZGlyZWN0b3JpZXMKICAgIAogICAg'
        'UmV0dXJuczoKICAgICAgICBQYXRoIG9iamVjdCBmb3IgdGhlIGRpcmVjdG9yeQogICAgIiIiCiAgICBkaXJfcGF0aCA9IFBh'
        'dGgocGF0aCkKICAgIAogICAgaWYgY3JlYXRlX3BhcmVudHM6CiAgICAgICAgZGlyX3BhdGgubWtkaXIocGFyZW50cz1UcnVl'
        'LCBleGlzdF9vaz1UcnVlKQogICAgZWxzZToKICAgICAgICBkaXJfcGF0aC5ta2RpcihleGlzdF9vaz1UcnVlKQogICAgCiAg'
        'ICByZXR1cm4gZGlyX3BhdGgKCgpkZWYgZ2V0X2ZpbGVfaGFzaChmaWxlX3BhdGg6IHN0cikgLT4gc3RyOgogICAgIiIiCiAg'
        'ICBDYWxjdWxhdGUgTUQ1IGhhc2ggb2YgYSBmaWxlLgogICAgCiAgICBBcmdzOgogICAgICAgIGZpbGVfcGF0aDogUGF0aCB0'
        'byBmaWxlCiAgICAKICAgIFJldHVybnM6CiAgICAgICAgTUQ1IGhhc2ggc3RyaW5nCiAgICAiIiIKICAgIGhhc2hfbWQ1ID0g'
        'aGFzaGxpYi5tZDUoKQogICAgCiAgICB3aXRoIG9wZW4oZmlsZV9wYXRoLCAncmInKSBhcyBmOgogICAgICAgIGZvciBjaHVu'
        'ayBpbiBpdGVyKGxhbWJkYTogZi5yZWFkKDQwOTYpLCBiIiIpOgogICAgICAgICAgICBoYXNoX21kNS51cGRhdGUoY2h1bmsp'
        'CiAgICAKICAgIHJldHVybiBoYXNoX21kNS5oZXhkaWdlc3QoKQoKCmRlZiBmb3JtYXRfZHVyYXRpb24oc2Vjb25kczogZmxv'
        'YXQpIC0+IHN0cjoKICAgICIiIgogICAgRm9ybWF0IGR1cmF0aW9uIGluIGh1bWFuLXJlYWRhYmxlIGZvcm0uCiAgICAKICAg'
        'IEFyZ3M6CiAgICAgICAgc2Vjb25kczogRHVyYXRpb24gaW4gc2Vjb25kcwogICAgCiAgICBSZXR1cm5zOgogICAgICAgIEZv'
        'cm1hdHRlZCBzdHJpbmcgKGUuZy4sICI1bSAzMHMiKQogICAgIiIiCiAgICBtaW51dGVzID0gaW50KHNlY29uZHMgLy8gNjAp'
        'CiAgICByZW1haW5pbmdfc2Vjb25kcyA9IGludChzZWNvbmRzICUgNjApCiAgICAKICAgIGlmIG1pbnV0ZXMgPiAwOgogICAg'
        'ICAgIHJldHVybiBmInttaW51dGVzfW0ge3JlbWFpbmluZ19zZWNvbmRzfXMiCiAgICBlbHNlOgogICAgICAgIHJldHVybiBm'
        'IntyZW1haW5pbmdfc2Vjb25kc31zIgoKCmRlZiBlc3RpbWF0ZV9nZW5lcmF0aW9uX3RpbWUoCiAgICB0b3RhbF9jbGlwczog'
        'aW50LAogICAgc2Vjb25kc19wZXJfY2xpcDogZmxvYXQgPSAxNS4wCikgLT4gZGljdDoKICAgICIiIgogICAgRXN0aW1hdGUg'
        'dG90YWwgZ2VuZXJhdGlvbiB0aW1lIGJhc2VkIG9uIGNsaXAgY291bnQuCiAgICAKICAgIEFyZ3M6CiAgICAgICAgdG90YWxf'
        'Y2xpcHM6IE51bWJlciBvZiBjbGlwcyB0byBnZW5lcmF0ZQogICAgICAgIHNlY29uZHNfcGVyX2NsaXA6IEF2ZXJhZ2UgdGlt'
        'ZSBwZXIgY2xpcCAoaW5jbHVkZXMgaW5mZXJlbmNlICsgb3ZlcmhlYWQpCiAgICAKICAgIFJldHVybnM6CiAgICAgICAgRGlj'
        'dGlvbmFyeSB3aXRoIHRpbWUgZXN0aW1hdGVzCiAgICAiIiIKICAgIHRvdGFsX3NlY29uZHMgPSB0b3RhbF9jbGlwcyAqIHNl'
        'Y29uZHNfcGVyX2NsaXAKICAgIAogICAgcmV0dXJuIHsKICAgICAgICAidG90YWxfY2xpcHMiOiB0b3RhbF9jbGlwcywKICAg'
        'ICAgICAiZXN0aW1hdGVkX3NlY29uZHMiOiB0b3RhbF9zZWNvbmRzLAogICAgICAgICJlc3RpbWF0ZWRfbWludXRlcyI6IHJv'
        'dW5kKHRvdGFsX3NlY29uZHMgLyA2MCwgMSksCiAgICAgICAgImZvcm1hdHRlZCI6IGZvcm1hdF9kdXJhdGlvbih0b3RhbF9z'
        'ZWNvbmRzKQogICAgfQoKCmRlZiBjaGVja19ncHVfYXZhaWxhYmlsaXR5KCkgLT4gZGljdDoKICAgICIiIgogICAgQ2hlY2sg'
        'R1BVIGF2YWlsYWJpbGl0eSBhbmQgY29uZmlndXJhdGlvbi4KICAgIAogICAgUmV0dXJuczoKICAgICAgICBEaWN0aW9uYXJ5'
        'IHdpdGggR1BVIGluZm9ybWF0aW9uCiAgICAiIiIKICAgIHJlc3VsdCA9IHsKICAgICAgICAiY3VkYV9hdmFpbGFibGUiOiBG'
        'YWxzZSwKICAgICAgICAiZ3B1X25hbWUiOiBOb25lLAogICAgICAgICJncHVfbWVtb3J5X2diIjogTm9uZSwKICAgICAgICAi'
        'ZGV2aWNlX2NvdW50IjogMAogICAgfQogICAgCiAgICB0cnk6CiAgICAgICAgaW1wb3J0IHRvcmNoCiAgICAgICAgCiAgICAg'
        'ICAgcmVzdWx0WyJjdWRhX2F2YWlsYWJsZSJdID0gdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKQogICAgICAgIAogICAgICAg'
        'IGlmIHJlc3VsdFsiY3VkYV9hdmFpbGFibGUiXToKICAgICAgICAgICAgcmVzdWx0WyJkZXZpY2VfY291bnQiXSA9IHRvcmNo'
        'LmN1ZGEuZGV2aWNlX2NvdW50KCkKICAgICAgICAgICAgCiAgICAgICAgICAgIGlmIHJlc3VsdFsiZGV2aWNlX2NvdW50Il0g'
        'PiAwOgogICAgICAgICAgICAgICAgcmVzdWx0WyJncHVfbmFtZSJdID0gdG9yY2guY3VkYS5nZXRfZGV2aWNlX25hbWUoMCkK'
        'ICAgICAgICAgICAgICAgIAogICAgICAgICAgICAgICAgIyBHZXQgbWVtb3J5IGluZm8KICAgICAgICAgICAgICAgIHRvdGFs'
        'X21lbW9yeSA9IHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9wcm9wZXJ0aWVzKDApLnRvdGFsX21lbW9yeQogICAgICAgICAgICAg'
        'ICAgcmVzdWx0WyJncHVfbWVtb3J5X2diIl0gPSByb3VuZCh0b3RhbF9tZW1vcnkgLyAoMTAyNCAqKiAzKSwgMSkKICAgIAog'
        'ICAgZXhjZXB0IEltcG9ydEVycm9yOgogICAgICAgIHBhc3MKICAgIAogICAgcmV0dXJuIHJlc3VsdAoKCmRlZiBnZXRfa2Fn'
        'Z2xlX2Vudmlyb25tZW50X2luZm8oKSAtPiBkaWN0OgogICAgIiIiCiAgICBHZXQgaW5mb3JtYXRpb24gYWJvdXQgdGhlIEth'
        'Z2dsZSBlbnZpcm9ubWVudC4KICAgIAogICAgUmV0dXJuczoKICAgICAgICBEaWN0aW9uYXJ5IHdpdGggZW52aXJvbm1lbnQg'
        'ZGV0YWlscwogICAgIiIiCiAgICBpbmZvID0gewogICAgICAgICJpc19rYWdnbGUiOiBGYWxzZSwKICAgICAgICAiZ3B1X3R5'
        'cGUiOiBOb25lLAogICAgICAgICJyYW1fZ2IiOiBOb25lLAogICAgICAgICJkaXNrX3NwYWNlX2diIjogTm9uZQogICAgfQog'
        'ICAgCiAgICAjIENoZWNrIGlmIHJ1bm5pbmcgb24gS2FnZ2xlCiAgICBpZiBvcy5wYXRoLmV4aXN0cygiL2thZ2dsZSIpOgog'
        'ICAgICAgIGluZm9bImlzX2thZ2dsZSJdID0gVHJ1ZQogICAgICAgIAogICAgICAgICMgVHJ5IHRvIGRldGVjdCBHUFUgdHlw'
        'ZSBmcm9tIGVudmlyb25tZW50CiAgICAgICAgZ3B1X2luZm8gPSBjaGVja19ncHVfYXZhaWxhYmlsaXR5KCkKICAgICAgICBp'
        'ZiBncHVfaW5mb1siY3VkYV9hdmFpbGFibGUiXToKICAgICAgICAgICAgaW5mb1siZ3B1X3R5cGUiXSA9IGdwdV9pbmZvWyJn'
        'cHVfbmFtZSJdCiAgICAgICAgICAgIGluZm9bInJhbV9nYiJdID0gZ3B1X2luZm9bImdwdV9tZW1vcnlfZ2IiXQogICAgICAg'
        'IAogICAgICAgICMgRXN0aW1hdGUgZGlzayBzcGFjZQogICAgICAgIHRyeToKICAgICAgICAgICAgaW1wb3J0IHNodXRpbAog'
        'ICAgICAgICAgICB0b3RhbCwgdXNlZCwgZnJlZSA9IHNodXRpbC5kaXNrX3VzYWdlKCIvIikKICAgICAgICAgICAgaW5mb1si'
        'ZGlza19zcGFjZV9nYiJdID0gcm91bmQoZnJlZSAvICgxMDI0ICoqIDMpLCAxKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246'
        'CiAgICAgICAgICAgIHBhc3MKICAgIAogICAgcmV0dXJuIGluZm8KCgpkZWYgdHJ1bmNhdGVfdGV4dCh0ZXh0OiBzdHIsIG1h'
        'eF9sZW5ndGg6IGludCA9IDEwMCwgc3VmZml4OiBzdHIgPSAiLi4uIikgLT4gc3RyOgogICAgIiIiCiAgICBUcnVuY2F0ZSB0'
        'ZXh0IHRvIG1heGltdW0gbGVuZ3RoLgogICAgCiAgICBBcmdzOgogICAgICAgIHRleHQ6IFRleHQgdG8gdHJ1bmNhdGUKICAg'
        'ICAgICBtYXhfbGVuZ3RoOiBNYXhpbXVtIGxlbmd0aAogICAgICAgIHN1ZmZpeDogU3VmZml4IHRvIGFkZCBpZiB0cnVuY2F0'
        'ZWQKICAgIAogICAgUmV0dXJuczoKICAgICAgICBUcnVuY2F0ZWQgdGV4dAogICAgIiIiCiAgICBpZiBsZW4odGV4dCkgPD0g'
        'bWF4X2xlbmd0aDoKICAgICAgICByZXR1cm4gdGV4dAogICAgCiAgICByZXR1cm4gdGV4dFs6bWF4X2xlbmd0aCAtIGxlbihz'
        'dWZmaXgpXSArIHN1ZmZpeAoKCmRlZiBzYW5pdGl6ZV9maWxlbmFtZShmaWxlbmFtZTogc3RyKSAtPiBzdHI6CiAgICAiIiIK'
        'ICAgIFNhbml0aXplIGEgc3RyaW5nIGZvciB1c2UgYXMgYSBmaWxlbmFtZS4KICAgIAogICAgQXJnczoKICAgICAgICBmaWxl'
        'bmFtZTogT3JpZ2luYWwgZmlsZW5hbWUKICAgIAogICAgUmV0dXJuczoKICAgICAgICBTYW5pdGl6ZWQgZmlsZW5hbWUKICAg'
        'ICIiIgogICAgIyBSZW1vdmUgb3IgcmVwbGFjZSBwcm9ibGVtYXRpYyBjaGFyYWN0ZXJzCiAgICBzYW5pdGl6ZWQgPSBmaWxl'
        'bmFtZS5zdHJpcCgpCiAgICAKICAgICMgUmVwbGFjZSBzcGFjZXMgd2l0aCB1bmRlcnNjb3JlcwogICAgc2FuaXRpemVkID0g'
        'c2FuaXRpemVkLnJlcGxhY2UoJyAnLCAnXycpCiAgICAKICAgICMgUmVtb3ZlIHNwZWNpYWwgY2hhcmFjdGVycwogICAgaW52'
        'YWxpZF9jaGFycyA9ICc8PjoiL1xcfD8qJwogICAgZm9yIGNoYXIgaW4gaW52YWxpZF9jaGFyczoKICAgICAgICBzYW5pdGl6'
        'ZWQgPSBzYW5pdGl6ZWQucmVwbGFjZShjaGFyLCAnJykKICAgIAogICAgIyBMaW1pdCBsZW5ndGgKICAgIGlmIGxlbihzYW5p'
        'dGl6ZWQpID4gMTAwOgogICAgICAgIHNhbml0aXplZCA9IHNhbml0aXplZFs6MTAwXQogICAgCiAgICByZXR1cm4gc2FuaXRp'
        'emVkIG9yICJ1bnRpdGxlZCIKCgpkZWYgcGFyc2VfZHVyYXRpb25fc3RyaW5nKGR1cmF0aW9uX3N0cjogc3RyKSAtPiBPcHRp'
        'b25hbFtmbG9hdF06CiAgICAiIiIKICAgIFBhcnNlIGEgZHVyYXRpb24gc3RyaW5nIGludG8gc2Vjb25kcy4KICAgIAogICAg'
        'U3VwcG9ydHMgZm9ybWF0cyBsaWtlOgogICAgLSAiNTozMCIgKDUgbWludXRlcyAzMCBzZWNvbmRzKQogICAgLSAiNW0gMzBz'
        'IgogICAgLSAiMzMwIiAoc2Vjb25kcykKICAgIC0gIjUuNSIgKG1pbnV0ZXMgYXMgZmxvYXQpCiAgICAKICAgIEFyZ3M6CiAg'
        'ICAgICAgZHVyYXRpb25fc3RyOiBEdXJhdGlvbiBzdHJpbmcKICAgIAogICAgUmV0dXJuczoKICAgICAgICBEdXJhdGlvbiBp'
        'biBzZWNvbmRzIG9yIE5vbmUgaWYgcGFyc2luZyBmYWlscwogICAgIiIiCiAgICB0cnk6CiAgICAgICAgIyBUcnkgTU06U1Mg'
        'Zm9ybWF0CiAgICAgICAgaWYgJzonIGluIGR1cmF0aW9uX3N0cjoKICAgICAgICAgICAgcGFydHMgPSBkdXJhdGlvbl9zdHIu'
        'c3BsaXQoJzonKQogICAgICAgICAgICBtaW51dGVzID0gaW50KHBhcnRzWzBdKQogICAgICAgICAgICBzZWNvbmRzID0gaW50'
        'KHBhcnRzWzFdKSBpZiBsZW4ocGFydHMpID4gMSBlbHNlIDAKICAgICAgICAgICAgcmV0dXJuIG1pbnV0ZXMgKiA2MCArIHNl'
        'Y29uZHMKICAgICAgICAKICAgICAgICAjIFRyeSAiWG0gWXMiIGZvcm1hdAogICAgICAgIGlmICdtJyBpbiBkdXJhdGlvbl9z'
        'dHIubG93ZXIoKToKICAgICAgICAgICAgcGFydHMgPSBkdXJhdGlvbl9zdHIubG93ZXIoKS5yZXBsYWNlKCdzJywgJycpLnNw'
        'bGl0KCdtJykKICAgICAgICAgICAgbWludXRlcyA9IGZsb2F0KHBhcnRzWzBdLnN0cmlwKCkpIGlmIHBhcnRzWzBdLnN0cmlw'
        'KCkgZWxzZSAwCiAgICAgICAgICAgIHNlY29uZHMgPSBmbG9hdChwYXJ0c1sxXS5zdHJpcCgpKSBpZiBsZW4ocGFydHMpID4g'
        'MSBhbmQgcGFydHNbMV0uc3RyaXAoKSBlbHNlIDAKICAgICAgICAgICAgcmV0dXJuIG1pbnV0ZXMgKiA2MCArIHNlY29uZHMK'
        'ICAgICAgICAKICAgICAgICAjIFRyeSBwbGFpbiBudW1iZXIgKGFzc3VtZSBzZWNvbmRzIGlmID4gNjAsIGVsc2UgbWludXRl'
        'cykKICAgICAgICB2YWx1ZSA9IGZsb2F0KGR1cmF0aW9uX3N0cikKICAgICAgICBpZiB2YWx1ZSA+IDYwOgogICAgICAgICAg'
        'ICByZXR1cm4gdmFsdWUgICMgQWxyZWFkeSBpbiBzZWNvbmRzCiAgICAgICAgZWxzZToKICAgICAgICAgICAgcmV0dXJuIHZh'
        'bHVlICogNjAgICMgVHJlYXQgYXMgbWludXRlcwogICAgCiAgICBleGNlcHQgKFZhbHVlRXJyb3IsIEluZGV4RXJyb3IpOgog'
        'ICAgICAgIHJldHVybiBOb25lCgoKZGVmIGJhdGNoX2l0ZXJhdGUoaXRlbXM6IExpc3RbQW55XSwgYmF0Y2hfc2l6ZTogaW50'
        'KToKICAgICIiIgogICAgSXRlcmF0ZSBvdmVyIGl0ZW1zIGluIGJhdGNoZXMuCiAgICAKICAgIEFyZ3M6CiAgICAgICAgaXRl'
        'bXM6IExpc3Qgb2YgaXRlbXMKICAgICAgICBiYXRjaF9zaXplOiBTaXplIG9mIGVhY2ggYmF0Y2gKICAgIAogICAgWWllbGRz'
        'OgogICAgICAgIEJhdGNoZXMgb2YgaXRlbXMKICAgICIiIgogICAgZm9yIGkgaW4gcmFuZ2UoMCwgbGVuKGl0ZW1zKSwgYmF0'
        'Y2hfc2l6ZSk6CiAgICAgICAgeWllbGQgaXRlbXNbaTppICsgYmF0Y2hfc2l6ZV0KCgpjbGFzcyBQcm9ncmVzc1RyYWNrZXI6'
        'CiAgICAiIiJTaW1wbGUgcHJvZ3Jlc3MgdHJhY2tlciBmb3IgbG9uZy1ydW5uaW5nIG9wZXJhdGlvbnMuIiIiCiAgICAKICAg'
        'IGRlZiBfX2luaXRfXyhzZWxmLCB0b3RhbDogaW50LCBkZXNjcmlwdGlvbjogc3RyID0gIlByb2dyZXNzIik6CiAgICAgICAg'
        'IiIiCiAgICAgICAgSW5pdGlhbGl6ZSBwcm9ncmVzcyB0cmFja2VyLgogICAgICAgIAogICAgICAgIEFyZ3M6CiAgICAgICAg'
        'ICAgIHRvdGFsOiBUb3RhbCBudW1iZXIgb2YgaXRlbXMKICAgICAgICAgICAgZGVzY3JpcHRpb246IERlc2NyaXB0aW9uIG9m'
        'IHRoZSBvcGVyYXRpb24KICAgICAgICAiIiIKICAgICAgICBzZWxmLnRvdGFsID0gdG90YWwKICAgICAgICBzZWxmLmN1cnJl'
        'bnQgPSAwCiAgICAgICAgc2VsZi5kZXNjcmlwdGlvbiA9IGRlc2NyaXB0aW9uCiAgICAKICAgIGRlZiB1cGRhdGUoc2VsZiwg'
        'YW1vdW50OiBpbnQgPSAxKSAtPiBOb25lOgogICAgICAgICIiIlVwZGF0ZSBwcm9ncmVzcyBieSBhbW91bnQuIiIiCiAgICAg'
        'ICAgc2VsZi5jdXJyZW50ICs9IGFtb3VudAogICAgICAgIHNlbGYuX2xvZygpCiAgICAKICAgIGRlZiBfbG9nKHNlbGYpIC0+'
        'IE5vbmU6CiAgICAgICAgIiIiTG9nIGN1cnJlbnQgcHJvZ3Jlc3MuIiIiCiAgICAgICAgcGVyY2VudCA9IChzZWxmLmN1cnJl'
        'bnQgLyBzZWxmLnRvdGFsICogMTAwKSBpZiBzZWxmLnRvdGFsID4gMCBlbHNlIDAKICAgICAgICBsb2dnZXIuaW5mbyhmIntz'
        'ZWxmLmRlc2NyaXB0aW9ufToge3NlbGYuY3VycmVudH0ve3NlbGYudG90YWx9ICh7cGVyY2VudDouMWZ9JSkiKQogICAgCiAg'
        'ICBkZWYgaXNfY29tcGxldGUoc2VsZikgLT4gYm9vbDoKICAgICAgICAiIiJDaGVjayBpZiBwcm9ncmVzcyBpcyBjb21wbGV0'
        'ZS4iIiIKICAgICAgICByZXR1cm4gc2VsZi5jdXJyZW50ID49IHNlbGYudG90YWwK'
    ),
    'presets/styles.yaml': (
        'cHJlc2V0czoKICBlcGljX29yY2hlc3RyYWw6CiAgICBiYXNlX3Byb21wdDogImVwaWMgb3JjaGVzdHJhbCwgZnVsbCBzeW1w'
        'aG9ueSwgY2luZW1hdGljLCBoYW5zIHppbW1lciBzdHlsZSIKICAgIG5lZ2F0aXZlX3Byb21wdDogImVsZWN0cm9uaWMsIHBv'
        'cCwgc2ltcGxlLCBsby1maSIKICAgIHRlbXBvX2JwbTogMTIwCiAgICBrZXk6ICJDIG1pbm9yIgogICAgaW5zdHJ1bWVudGF0'
        'aW9uOiAic3RyaW5ncywgYnJhc3MsIHRpbXBhbmksIGNob2lyIgogIAogIHN5bnRod2F2ZV9yZXRybzoKICAgIGJhc2VfcHJv'
        'bXB0OiAiODBzIHN5bnRod2F2ZSwgcmV0cm8gYW5hbG9nIHN5bnRocywgZHJpdmluZyBiYXNzbGluZSwgZHJ1bSBtYWNoaW5l'
        'IgogICAgbmVnYXRpdmVfcHJvbXB0OiAiYWNvdXN0aWMsIG9yY2hlc3RyYWwsIHZvY2FscyIKICAgIHRlbXBvX2JwbTogMTEw'
        'CiAgICBrZXk6ICJGIG1pbm9yIgogICAgaW5zdHJ1bWVudGF0aW9uOiAiYW5hbG9nIHN5bnRocywgZ2F0ZWQgZHJ1bXMsIGJh'
        'c3Mgc3ludGgiCiAgCiAgbG9maV9oaXBob3A6CiAgICBiYXNlX3Byb21wdDogImxvZmkgaGlwIGhvcCwgY2hpbGwgYmVhdHMs'
        'IHZpbnlsIGNyYWNrbGUsIHJlbGF4ZWQgZ3Jvb3ZlIgogICAgbmVnYXRpdmVfcHJvbXB0OiAibG91ZCwgYWdncmVzc2l2ZSwg'
        'ZWxlY3Ryb25pYyBkYW5jZSIKICAgIHRlbXBvX2JwbTogODUKICAgIGtleTogIkEgbWFqb3IiCiAgICBpbnN0cnVtZW50YXRp'
        'b246ICJib29tIGJhcCBkcnVtcywgUmhvZGVzIHBpYW5vLCBiYXNzIGd1aXRhciIKICAKICBhbWJpZW50X2VsZWN0cm9uaWM6'
        'CiAgICBiYXNlX3Byb21wdDogImFtYmllbnQgZWxlY3Ryb25pYywgYXRtb3NwaGVyaWMgcGFkcywgZXRoZXJlYWwgdGV4dHVy'
        'ZXMsIHNsb3cgZXZvbHZpbmciCiAgICBuZWdhdGl2ZV9wcm9tcHQ6ICJyaHl0aG1pYywgcGVyY3Vzc2l2ZSwgdm9jYWxzIgog'
        'ICAgdGVtcG9fYnBtOiA3MAogICAga2V5OiAiRCBtYWpvciIKICAgIGluc3RydW1lbnRhdGlvbjogInN5bnRoIHBhZHMsIGZp'
        'ZWxkIHJlY29yZGluZ3MsIHNvZnQgYmVsbHMiCiAgCiAgcm9ja19hbHRlcm5hdGl2ZToKICAgIGJhc2VfcHJvbXB0OiAiYWx0'
        'ZXJuYXRpdmUgcm9jaywgZWxlY3RyaWMgZ3VpdGFycywgZHluYW1pYyBkcnVtcywgZW5lcmdldGljIgogICAgbmVnYXRpdmVf'
        'cHJvbXB0OiAib3JjaGVzdHJhbCwgZWxlY3Ryb25pYywgc29mdCIKICAgIHRlbXBvX2JwbTogMTMwCiAgICBrZXk6ICJFIG1p'
        'bm9yIgogICAgaW5zdHJ1bWVudGF0aW9uOiAiZGlzdG9ydGVkIGd1aXRhcnMsIGJhc3MsIGxpdmUgZHJ1bXMiCiAgCiAgamF6'
        'el9zbW9vdGg6CiAgICBiYXNlX3Byb21wdDogInNtb290aCBqYXp6LCBzYXhvcGhvbmUsIHVwcmlnaHQgYmFzcywgYnJ1c2hl'
        'ZCBkcnVtcywgc29waGlzdGljYXRlZCIKICAgIG5lZ2F0aXZlX3Byb21wdDogImVsZWN0cm9uaWMsIHJvY2ssIGFnZ3Jlc3Np'
        'dmUiCiAgICB0ZW1wb19icG06IDk1CiAgICBrZXk6ICJCYiBtYWpvciIKICAgIGluc3RydW1lbnRhdGlvbjogInNheG9waG9u'
        'ZSwgcGlhbm8sIHVwcmlnaHQgYmFzcywgYnJ1c2hlcyIKICAKICBjaW5lbWF0aWNfdHJhaWxlcjoKICAgIGJhc2VfcHJvbXB0'
        'OiAiY2luZW1hdGljIHRyYWlsZXIgbXVzaWMsIGVwaWMgaGl0cywgcmlzaW5nIHRlbnNpb24sIGRyYW1hdGljIHN0cmluZ3Mi'
        'CiAgICBuZWdhdGl2ZV9wcm9tcHQ6ICJzb2Z0LCBxdWlldCwgbWluaW1hbGlzdCIKICAgIHRlbXBvX2JwbTogMTQwCiAgICBr'
        'ZXk6ICJDIG1pbm9yIgogICAgaW5zdHJ1bWVudGF0aW9uOiAib3JjaGVzdHJhbCBoaXRzLCBoeWJyaWQgc3ludGhzLCBwZXJj'
        'dXNzaW9uIgogIAogIHBvcF91cGJlYXQ6CiAgICBiYXNlX3Byb21wdDogInVwYmVhdCBwb3AsIGNhdGNoeSBtZWxvZGllcywg'
        'YnJpZ2h0IHByb2R1Y3Rpb24sIHJhZGlvIGZyaWVuZGx5IgogICAgbmVnYXRpdmVfcHJvbXB0OiAiZGFyaywgc2xvdywgZXhw'
        'ZXJpbWVudGFsIgogICAgdGVtcG9fYnBtOiAxMjUKICAgIGtleTogIkcgbWFqb3IiCiAgICBpbnN0cnVtZW50YXRpb246ICJz'
        'eW50aHMsIHByb2dyYW1tZWQgZHJ1bXMsIGJhc3MsIHZvY2FsIGNob3BzIgoKIyBTZWN0aW9uLXNwZWNpZmljIGVuZXJneSBt'
        'b2RpZmllcnMKc2VjdGlvbl9tb2RpZmllcnM6CiAgaW50cm86CiAgICBlbmVyZ3lfbGV2ZWw6IDAuNgogICAgcHJvbXB0X2Fk'
        'ZGl0aW9uczogImJ1aWxkaW5nIHVwLCBpbnRyb2R1Y3RvcnksIHNldHRpbmcgdGhlIG1vb2QiCiAgICBkdXJhdGlvbl9yYXRp'
        'bzogMC4xCiAgdmVyc2U6CiAgICBlbmVyZ3lfbGV2ZWw6IDAuNwogICAgcHJvbXB0X2FkZGl0aW9uczogInN0ZWFkeSByaHl0'
        'aG0sIGZvY3VzZWQsIG5hcnJhdGl2ZSBmZWVsIgogICAgZHVyYXRpb25fcmF0aW86IDAuMjUKICBjaG9ydXM6CiAgICBlbmVy'
        'Z3lfbGV2ZWw6IDAuOTUKICAgIHByb21wdF9hZGRpdGlvbnM6ICJoaWdoIGVuZXJneSwgZnVsbCBpbnN0cnVtZW50YXRpb24s'
        'IHNvYXJpbmcsIGFudGhlbWljIgogICAgZHVyYXRpb25fcmF0aW86IDAuMjUKICBicmlkZ2U6CiAgICBlbmVyZ3lfbGV2ZWw6'
        'IDAuOAogICAgcHJvbXB0X2FkZGl0aW9uczogInNoaWZ0aW5nIHRvbmUsIGJ1aWxkaW5nIHRlbnNpb24sIHRyYW5zaXRpb25h'
        'bCIKICAgIGR1cmF0aW9uX3JhdGlvOiAwLjE1CiAgb3V0cm86CiAgICBlbmVyZ3lfbGV2ZWw6IDAuNQogICAgcHJvbXB0X2Fk'
        'ZGl0aW9uczogImZhZGluZyBvdXQsIHJlc29sdmluZywgY29uY2x1c2l2ZSIKICAgIGR1cmF0aW9uX3JhdGlvOiAwLjEK'
    ),
}

for _rel, _b64 in _BUNDLE.items():
    _dest = pathlib.Path(NOTEBOOK_DIR) / _rel
    _dest.parent.mkdir(parents=True, exist_ok=True)
    _dest.write_bytes(base64.b64decode(''.join(_b64) if isinstance(_b64, tuple) else _b64))

# The *parent* goes on the path, not src/ itself: the modules import each other relatively
# (`from .song_arranger import ...`), which needs them to be a package rather than a set of
# top-level modules. src/__init__.py makes `src` that package.
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)

from src.preset_engine import PresetEngine
from src.song_arranger import SongArranger
from src.long_form_generator import LongFormGenerator
from src.audio_stitcher import stitch_song_sections
from src.api_server import app
from src.tunnel_manager import TunnelManager

print(f'✅ Source package written to {NOTEBOOK_DIR} and imported')
print(f'   Available presets: {PresetEngine().list_presets()}')


## Step 4: Check GPU Environment

In [ ]:
import torch

print("=== GPU Environment ===")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"CUDA Version: {torch.version.cuda}")
else:
    print("⚠️ No GPU detected - generation will be very slow")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nUsing device: {device}")

## Step 5: Start FastAPI Server with Tunnel

In [ ]:
import threading
import time
import uvicorn
from src.tunnel_manager import TunnelManager

# Configuration
PORT = 8000
OUTPUT_DIR = "/kaggle/working/riffusion_output"

# Initialize tunnel manager
tunnel_mgr = TunnelManager()

# Start the API server in a background thread
def run_server():
    uvicorn.run(
        "src.api_server:app",
        host="0.0.0.0",
        port=PORT,
        log_level="info"
    )

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Wait for server to start
time.sleep(2)

# Start cloudflared tunnel
print("Starting Cloudflare tunnel...")
success, tunnel_url = tunnel_mgr.start_tunnel(PORT)

if success and tunnel_url:
    print("\n" + "="*60)
    print("🎉 SERVER READY!")
    print("="*60)
    print(f"\n📡 Public API URL: {tunnel_url}")
    print(f"\n💡 Usage:")
    print(f"   GET  {tunnel_url}/health - Health check")
    print(f"   GET  {tunnel_url}/presets - List presets")
    print(f"   POST {tunnel_url}/generate_song - Generate a song")
    print(f"   GET  {tunnel_url}/status/{{job_id}} - Check status")
    print(f"   GET  {tunnel_url}/download/{{job_id}} - Download result")
    print("\n" + "="*60)
    print("⏳ Keep this notebook running to maintain the server")
    print("="*60)
else:
    print("❌ Failed to start tunnel")

## Step 6: Test the API (Optional)

In [ ]:
# Test the API is responding
import requests

if tunnel_url := tunnel_mgr.get_tunnel_url():
    # Health check
    response = requests.get(f"{tunnel_url}/health")
    print(f"Health check: {response.json()}")
    
    # List presets
    response = requests.get(f"{tunnel_url}/presets")
    print(f"\nAvailable presets: {response.json()['presets']}")
    
    # Get API key
    response = requests.get(f"{tunnel_url}/api-key")
    api_key = response.json()['api_key']
    print(f"\nAPI Key: {api_key}")
    print("\n✅ API is working!")
else:
    print("Tunnel not started yet")

## Example: Generate a Song via API

In [ ]:
# Example usage (uncomment to test)
# 
# import requests
# 
# if tunnel_url := tunnel_mgr.get_tunnel_url():
#     payload = {
#         "title": "My AI Song",
#         "lyrics": """
#         [Verse 1]
#         In the digital dreams we weave
#         Through the code we believe
#         
#         [Chorus]
#         Singing loud for all to hear
#         The future of music is here
#         """,
#         "style_preset": "synthwave_retro",
#         "target_duration_minutes": 5.0
#     }
#     
#     response = requests.post(f"{tunnel_url}/generate_song", json=payload)
#     job_data = response.json()
#     print(f"Job created: {job_data}")
#     
#     # Poll for completion
#     job_id = job_data['job_id']
#     while True:
#         status_resp = requests.get(f"{tunnel_url}/status/{job_id}")
#         status = status_resp.json()
#         print(f"Status: {status['status']} - {status.get('progress', {})}")
#         
#         if status['status'] in ['completed', 'failed', 'cancelled']:
#             break
#         
#         time.sleep(10)
#     
#     # Download result
#     if status['status'] == 'completed':
#         download_resp = requests.get(f"{tunnel_url}/download/{job_id}")
#         with open(f"/kaggle/working/{job_id}.wav", 'wb') as f:
#             f.write(download_resp.content)
#         print(f"Song saved to /kaggle/working/{job_id}.wav")